In [148]:
from brian2 import *
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.signal import windows, butter, filtfilt, resample
from scipy.stats import linregress
import numpy as np
import seaborn as sns
from sklearn.decomposition import PCA
import random
import matplotlib.cm as cm
import os
import pandas as pd
import sys
from factor_analyzer import FactorAnalyzer
import scipy.io

In [149]:
# Used to convert a string variable (used as input from the iteration script) to a boolean variable (when calling this script with specific inputs as strings)
def str_to_bool(s):
    if s == "True":
        return True
    elif s == "False":
        return False
    else:
        return False

In [150]:
start_scope()  # Re-initialize Brian
np.random.seed(100)  # For reproducibility

### GENERAL PARAMETERS ###########################################################
sim_name = "$_Three_heterogeneously_distributed_inhibitory_inputs" # str(os.getenv('simulation_name')) # < to use when iterating the simulation via another script # '___TEST_excit_distrib_small1_large1_relationship1' 
sim_method = 'euler' # 'exact' 'euler' #'euler' is less precise but faster, and seems to be good enough
pool_colors_one = ['C0','C1','C2']
pool_colors_two = ['green','purple','gold']
pool_cmap = ['viridis','plasma','autumn']

# ANALYSIS PARAMETERS
factor_analysis_on_all_pools = True # If true, all motor neurons from all pools will be used for FA. If false, only the first pool will be considered
# To be aware, if the duration of the simulation is low compared to the number of motor neurons (many features)
skip_per_MU_force_computation = True # Faster when disabled
save_binary_dsicharge_mat_as_csv = False # Takes up some memory storage space

# TIME PARAMETERS
fsamp = 1000  # set your fsamp # This is NOT the dt at which the simulation runs. The simulation timesteps are 0.1ms in duration by default
true_duration = 60 # 20 # 20 # 60 # in s
window_beginning_ignore = 1 # in s
window_end_ignore = 1 # in s
duration_with_ignored_window = (true_duration+window_beginning_ignore+window_end_ignore)*second
ISI_threshold_for_discontinuity = 0.4 # np.inf # 0.4 # in s ; motoneurons whose max(ISI)>threshold will be removed from analysis (so only continuous MNs are kept)
samples_to_consider = np.arange(window_beginning_ignore*fsamp,(duration_with_ignored_window/second-window_end_ignore)*fsamp).astype(int) # samples corresponding to the time window for which the analysis is done (ignoring window_beginning_ignore & window_end_ignore)

# NUMBER OF NEURONS SIMULATED
nb_pools = 1 # Number of pools of motoneurons to simulate
nb_motoneurons_per_pool = 300 # Nb of motor neurons per pool # Need at least 2, or N with N*Ib_inh_int_to_MN_connectivity_probability>=1
total_nb_motoneurons = nb_motoneurons_per_pool*nb_pools
# Rough approximation of the number of motor units found in the tibialis anterior in humans (Motor Unit - Heckamn & Enoka 2012, ref 220 & 694)
# Twitch torque data also comes from the tibialis anterior, so this hopefully allows for a realistic simulation of the motor pool behavior for a given force level
nb_Renshaw_cells_per_pool = 60 # Need at least N, with N*MN_to_Renshaw_connectivity_probability_within_pool >= 1
# Williams & Baker 2009 simulation = 377 MN and 64 Renshaw cells (~ratio of 5/1)
total_nb_Ib_interneurons= 60 # Need at least 1

# VOLTAGE THRESHOLDS OF ALL NEURONS
voltage_rest = 0 * mvolt # arbitrary ; 0 at rest
voltage_thresh = 10 * mvolt # arbitrary ; 10 for generating a spike

# REVERSAL/EQUILIBRIUM POTENTIAL OF LEAK CHANNELS, EXCITATORY CHANNELS, INHIBITORY CHANNELS
E_leak = voltage_rest
# Reversal potentials (relative to resting potential) from Elias & Kohn 2013 are 70 and -16 for xcitatory and inhibitory, respectively.
# They corrspond roughly to the reversal potentials of sodium channels (for E_excit) and chloride channels (for E_inhib).
# However, to make everything easier to work with, I tried to make the inhibition and excitation have roughly equivalent net effects on firing rates by making the reversal potential symmetric relative to half the firing threshold.
E_excit = ((voltage_thresh + voltage_rest)/2) + 20 * mvolt # Reversal potential for excitatory input
E_inhib = ((voltage_thresh + voltage_rest)/2) - 20 * mvolt  # Reversal potential for inhibitory input


### INPUT PARAMETERS #########################################################

# Excitatory common input
excitatory_input_baseline = 75*1e-2 # 50*1e-2 allows to get a mean firing rate of valid MUs of ~9-10pps, and the biggest valid MU has a relative size of 0.2
# +10 when inhibitory input

nb_excitatory_inputs_per_pool = 1

excitatory_inputs_std = 3*1e-2
low_pass_filter_of_excitatory_input = 2.5 # in hz
excitatory_input_source = 'load_experimental_data' # 'generate_synthetic_input' ; 'load_experimental_data' ; 'generate_burst_input' # 'generate_burst_input' is used to test force generation
excitatory_input_sourcefile_path = "C:\\Users\\fdernoncourt\\Documents\\Simulation_input_files" # used only if excitatory_input_source == 'load_experimental_data'
excitatory_input_sourcefile_filename = "\\S2_VL_DEFr_input_for_simulation.mat" # str(os.getenv('input_data_filename')) # "\S1_VL_HUFr_input_for_simulation.mat"
excitatory_input_sourcefile_fsamp = 2048
# If loadind experimental data, distribute the PCs over the total of excitatory inputs in order
# For example, if 2 pools and 2 inputs per pool:
# PC1 for pool0,input0; PC2 for pool0,input1; PC3 for pool1,input0; PC4 for pool1,input1

set_same_excitatory_input_for_all_pools = False # If true, this will override all the generated excitatory inputs to be the same for all pools
set_arbitrary_correlation_between_excitatory_inputs = False # If true but only one input in total, will return an error
within_pool_excitatory_input_correlation = 0 # > 0 and < 1 # used only if set_arbitrary_correlation_between_excitatory_inputs == True
between_pool_excitatory_input_correlation = 0.7 # > 0 and < 1 # used only if set_arbitrary_correlation_between_excitatory_inputs == True
# negative correlations are accepted as inputs but the procedure to set arbitrary correlations doesn't work for negative correlations (it results in correlations around 0)
# a rough method has been implemented to deal with this problem, but it can induce spurious reversal of the sign of correlations

distribution_of_excitation = 'heterogeneous_constant_sum_of_weights' # 'heterogeneous_constant_sum_of_weights', 'heterogeneous_random_sum_of_weights', 'homogeneous_constant_sum_of_weights'


# Inhibitory common input
nb_inhibitory_inputs_per_pool = 3

inhibitory_input_mean = 12*1e-2 # 10*1e-2 # float(os.getenv('inhibitory_input_mean')) < to use when iterating the simulation via another script # in millisiemens
if nb_inhibitory_inputs_per_pool == 0:
    inhibitory_input_mean = 0 # the inhibitory input mean is set as a baseline for all motor units in the presence of inhibition, with additional inhibitory input simply being zero-mean signals added on top. So, if no inhibition, inhibition mean is set to 0
inhibitory_input_std = 3*1e-2 # 5*1e-2 # float(os.getenv('inhibitory_input_std'))# in millisiemens
low_pass_filter_of_inhibitory_input = 2.5 #in hz

distribution_of_inhibition = 'heterogeneous_constant_sum_of_weights' # 'heterogeneous_constant_sum_of_weights', 'heterogeneous_random_sum_of_weights', 'homogeneous_constant_sum_of_weights'

set_arbitrary_correlation_between_inhibitory_and_excitatory_inputs = False
within_pool_excitatory_to_inhibitory_input_correlation = -0.5 # relative to the first common excitatory input that shares the same pool as the inhibitory input
between_pool_excitatory_to_inhibitory_input_correlation = 0.5 # relative to the first common excitatory input that doesn't share the same pool as the inhibitory input
# Please note that the setting of the correlation between excitatory and inhibitory inputs does not work very well, especially with multiple pools

# Independent noise (both excitatory and inhibitory) inputs - for all cell types
low_pass_filter_of_MN_independent_noise = 50 # in Hz
MN_independent_noise_excit_std = excitatory_inputs_std*3 # 18*1e-2 # in millisiemens
MN_independent_noise_inhib_std = inhibitory_input_std*3 # 0*1e-2 # 15*1e-2 # in millisiemens
# The other cell types are not conductance-based
low_pass_filter_of_other_cell_types_independent_noise = 50 # in Hz
RC_independent_noise_std = 3 # 1 # in mV
Ib_interneruons_noise_std = 3 # 1 # in mV

## MOTOR UNIT (MOTOR NEURONS AND ASSOCIATED MUSCLE FIBERS) PARAMETERS ##########################################################################

# DISTRIBUTION OF MOTONEURON SIZES
min_soma_diameter = 50 # in micrometers, for smallest MN
max_soma_diameter = 100 # in micrometers, for largest MN
# Assuming that soma diameter from human motoneurons vary between 50 and 100 micrometers, loosely based on https://journals.physiology.org/doi/full/10.1152/physiol.00021.2018 (mean diameter of humans MN estimated to be ~60 micrometers)
    # ^ "Scaling of motoneurons, From Mouse to Human" Manuel et al. Physiology (2018)
# Parameter to create an exponetially decreasing dsitribution curve, with larger motoneurons being less numerous than smaller motoneurons
# Somewhat fitting the curve in Principles of Neural Science 2021 edition, Enoka chapter on motor units, fig 31-3.A
size_distribution_exponent = 2
# between 0-1 => more large MN than small MN; 1 => uniform distribution (linear relationship between MN index and soma diameter); >1 => more small motoneurons than laarge MNs
# VIsualize distribution for different min and max soma diameters, and different exponents = https://www.desmos.com/calculator/zy3ywcz4tn 

#### ELECTROPHYSIOLOGICAL MN PROPERTIES #####
# Electrophysiological properties calculated from Caillet et al 2022 https://elifesciences.org/articles/76489
# TIME CONSTANT - MILLISECONDS
tau_MN_constant = 2.6*(10**4) # Caillet et al 2022
tau_MN_exponent = 1.5 # Caillet et al 2022
# Not calculated directly from Capacitance and Resistance but from Caillet's equations, which are meant to better fit experimental data even though they do not follow the exact rule tau = R*C
# RESISTANCE - OHMS
# The resistance decreases the leak conductance and increases the weight of the excitatory and inhibitory input received by the motor neuron (=> higher resistance means higher sensitivity to input)
resistance_constant = 9.6*(10**5) # Caillet et al 2022 # in Ohms
resistance_exponent = 2.4*(-1) # Caillet et al 2022 # in Ohms
# https://www.desmos.com/calculator/pbs97zynff = visualize the curve for resistance (ohms) and input weights (between 0 and 1)
# Min resistance for smallest MN (50 micrometers) = ~80 ohms
# Max resistance for biggest MN (100 micrometers) = ~15 ohms
#### Input weight = normalized resistance, so that the input to the smallest MN is scaled by a factor of 1 #####
# Min input weight for smallest MN (50 micrometers) = 1
# Max input weight for biggest MN (100 micrometers) = ~0.19
# CONDUCTANCE - SIEMENS
membrane_conductance_scaling = 1 # membrane conductance is 1/resistance, multiplied by a scalar value (tuned by hand) to get realistic behavior of MN pool
# RHEOBASE - AMPERES
# The rheobase is modeled as an offset to the change in excitatory conductance caused by the exitatory input (clamped to 0 to avoid the excitatory input to have an hyperpolarization effect)
rheobase_constant = 9.0*(10**-4) # Caillet et al 2022 # in nanoAmps
rheobase_exponent = 2.5 # Caillet et al 2022 # in nanoAmps
rheobase_scaling = 100 # float(os.getenv('rheobase_scaling')) # < to use when iterating the simulation via another script # 100 # scalar value to multiply the rheobase by, tuned to get realistic behavior according to the arbitrary values used (such as voltage threshold and reversal potentials)
# CAPACITANCE - FARADS
capacitance_constant = 1.2 # Caillet et al 2022
capacitance_exponent = 1 # Caillet et al 2022
# AFTERHYPERPOLARIZATION DURATION & REFRACTORY PERIOD - SECONDS
# Refractory period (Caillet's paper gives equations for AHP duration but not for refractory period duration) #
# Since we are using a simplified model, we approximate the effect of the AHP by implementing an absolute refactory period that is a fraction of the true AHP duration.
AHP_duration_constant = 2.5 * (10**4) # Caillet et al 2022
AHP_duration_exponent = 1.5 * (-1) # Caillet et al 2022
refractory_period_as_AHP_fraction = 0.2 # float(os.getenv('refractory_period_as_AHP_fraction')) # Manually tuned
    # Manuel et al. 2019 "Scaling of motor output, from Mouse to Humans"
        # "Statistical methods employed at low firing rates indicate the AHP durations of low-threshold human motoneurons, presumably type S and perhaps some type FR, are ~125–140 ms."
    # Herbert & Gandevia 1999 assume a 5ms (absolute?) refractory period
    # Lateva et at 2001 = Absolute refractory period of 3ms in muscle fibers, and relative refractory period of 10ms
    # University of Washington textbook of physiology = in a typical neuron, the absolute refractory period lasts a few ms and the relative period tens of ms

### CONTRACTILE PROPERTIES (TWITCH FORCE CAUSED BY MN SPIKES)

fiber_force_rest = 0 * volt # Resting force
force_twitch_rise_tau_baseline = 20*ms # in ms # rise of twitch force
force_decay_tau_baseline = 50*ms #in ms # decay of twitch force
force_baseline_conductance_scalar = 5*siemens # conductance for the force decay (return to rest)
### VALUES RELATIVE TO MU SIZE
# Those values are linearly interpolated according to MN size
smallest_MU_tau_multiplier = 1.5 # constant to multiply the twitch rise and decay tau by to get the contraction speed of the smallest MU. The larger the value, the slower the contraction speed
largest_MU_tau_multiplier = 0.75 # constant to multiply the twitch rise and decay tau by to get the contraction speed of the largest MU. The lower the value, the higher the contraction speed
smallest_MU_max_force = 1 # arbitrary units # Maximum (tetanus) force of muscle fiber = "Equilibrium potential of the force conductance" (internally, considered as voltage by Brian2)
largest_MU_max_force = 10  # arbitrary units  # Maximum (tetanus) force of muscle fiber = "Equilibrium potential of the force conductance" (internally, considered as voltage by Brian2)
    # The force is normalized to the maximal muscle force (sum of the tetanus force of all fibers), so only the ratio of largest MU force over smallest MU force matters
### ELECTROMECHANICAL DELAY - SECONDS ; AXONAL CONDUCTION VELOCITY - METERS/SECOND
# Electromechanical delay => inverse of actional conduction velocity
# Calculated from the axonal conduction velocity relationship reported in Caillet et al 2022
# ^ multiplying by two (assuming a 0.5m axon length => so correspond to the conduction speed from MN to muscle fiber)
axonal_conduction_velocity_constant = 4.0*2 # Caillet et al 2022
axonal_conduction_velocity_exponent = 0.7 # Caillet et al 2022


## RENSHAW CELLS AND IB INTERNEURONS PARAMETERS ##########################################################################

### Renshaw cells electrophysiological parameters
tau_Renshaw = 8*ms # time constant of Renshaw cells
# Williams & Baker 2009: "time constant of 8ms, similar to experimental data (Desilligny, 1979; Hultborn et al., 1979)"
# Maltenfort, Heckman 1998 simulation study = 8ms time constant and 30ms AHP
refractory_period_RC = 10*ms # refractory period of Renshaw cells => Williams & Baker 2009: "AHP of 36ms similar to experimental data (Deseilligny, 1979; Hultborn et al., 1979)"
# Firing rates of RCs can be expected to go up to >60pps (Moore et al 2015)
# Maltenfort, Heckman 1998 = "The maximum dendrites that happen to travel into the motor nucleus region. In steady-state firing rate of Renshaw cells is 200 pps (Cleveland et al. 1981)"

### Ib inhiitory interneurons electrophysiological parameters
tau_Ib_interneurons = 8*ms # time constant of Renshaw cells
refractory_period_Ib_interneurons = 10*ms # refractory period of Renshaw cells => Williams & Baker 2009: "AHP of 36ms similar to experimental data (Deseilligny, 1979; Hultborn et al., 1979)"


## CONNECTIVITY ##########################################################################

# Motor neurons and Renshaw cells connectivity
MN_to_Renshaw_connectivity_probability_within_pool = 0.17 #0.2 # 0.5 # 0.25 #if 0.1, each Renshaw cell will receive excitatory input from a random subset of 10% of the homonymous MN pool
Renshaw_to_MN_connectivity_probability_within_pool = 0.4 #0.6 # 0.25 # 0.5 #if 0.3, each Renshaw cell will send inhibitory input to a random subset of 30% of the homonymous MN pool
    # Williams & Baker 2009 = " each motoneuron receives input from 10-20 Renshaw cells, and each Renshaw cell receives input from 20-50 motoneurons."
        # => Motoneurons receive input from [10 to 20]/64 Renshaw cells (0.16 to 0.31), each Renshaw cell receives input from [20-50]/377 MNs  (0.05 to 0.13)
    # From Moore et al 2015: a typical Renshaw cell receives input from ~6-7 motoneurons, and that a Renshaw cell projects back to ~40 motoneurons (so 40/6 ratio )
        # "The results of the present study indicate that the number of contacts from a motoneuron (7.1 +- 1.2) is indeed 6. (...)
        # (...) however, is less than the extrapolated count reported previously (Alvarez et al., 1999). (...)
        # Our estimates of both the number of contacts from all motoneurons and of the convergence of 4 motoneurons contacting individual Renshaw cells can therefore only represent lower bounds.
        # Previous paired recordings (Bhumbra et al., 2014) of the Renshaw cell to motoneuron synapse report an average of 5.5+-0.5 for the number of contacts.
        # The results from the ventral root stimulation experiments of the present study yields an average of 225 release sites, suggesting a convergence quotient of 40 Renshaw cells per motoneuron.
        # These estimates again represent lower bounds because of the slice preparation.
        # Within the limits highlighted above, our data suggest that the degree of convergence for the inhibitory projection may be as much as 10 times greater than that of the excitatory projection."
    # => Ratio of MN->Renshaw and Renshaw->MN comprised between 1/3 and 1/10
    # Edgley, Williams, Baker 2021 = proportion is very different across muscles anyway (primate upper limb)
    # Maltenfort, Heckman 1998 =
        # Simulated probability of connectivity between RCs and MNs according to distance. Max distance (aribitrary) of 15 for connections from RC to MNs,
        # and max distance of 2 for connections from MNs to RC
        # "Each simulated motoneuron therefore synapsed on five Renshaw cells (...) each Renshaw cell could receive input from 20 motoneurons"
        # ^ 256 MNs simulated, so ~0.1 ratio
MN_to_Renshaw_connectivity_probability_across_pool = 0 # 0.1 #0.1 #if 0.1, each Renshaw cell will receive excitatory input from a random subset of 10% of the MNs from the other pools
Renshaw_to_MN_connectivity_probability_across_pool = 0 # 0.1 # 0.3 #0.3 #if 0.3, each Renshaw cell will send inhibitory input to a random subset of 30% of the MNs from the other pools

MN_to_Renshaw_connectivity_prevent_heteronymous_pool_overlap = False # if True, a Renshaw cell cannot receive input from MNs from several heteronymous pools
Renshaw_to_MN_connectivity_prevent_heteronymous_pool_overlap = False # if True, a MN cannot receive input from Renshaw cells from from several heteronymous pools


# Ib interneurons to motor neurons connectivity
Ib_inh_int_to_MN_connectivity_probability = 0.4
Ib_inh_int_to_MN_connectivity_gaussian_nonbinary_weights = False # if true, mean weight = Ib_inh_int_to_MN_connectivity_probability, std defined below
Ib_inh_int_to_MN_connectivity_probability_std = 0.15 # used only if Ib_inh_int_to_MN_connectivity_gaussian_nonbinary_weights = True


### POST-SYNAPTIC EFFECTS ##########################
MN_to_Renshaw_excit = 0*mvolt # 0*mvolt # 6.7*mvolt # in mV # increase in V in renshaw cell when receiving spike from MN - From Moore et al 2015 = MN-RC pair recordings, with 1 MN spike on average resulting in a probability of 0.3 of RC spike
Renshaw_to_MN_inhib = 0*1e-2*msiemens # 0*siemens # 5*1e-2*siemens  # in siemens # Increase in inhibitory conductance in MN when receiving spike from Renshaw cell
MN_RC_synpatic_delay = 5*ms # in both directions # Williams & Baker 2009: "A 1 ms conduction delay was introduced for both motoneuron to Renshaw cell, and Renshaw cell to motoneuron contacts."
# https://pmc.ncbi.nlm.nih.gov/articles/PMC3008340/ Mentis 2006 J Neuroscience = ~7ms delay
# https://www.cell.com/neuron/fulltext/S0896-6273(18)30789-X Hoang 2018 Neuron, figure 7 = mean ~5ms delay
MN_to_muscle_fiber_force = 6 # increase in "force conductance" (makes it move towards tetanic force) in muscle fiber when receiving spike from MN # scalar that will multiply the "baseline force conductance"
force_to_Ib_excit = 0*mvolt # scalar # how much 1% of MVC raises the voltage of the Ib interneuron # as it is implemented now, it depends heavily on the number of MNs
Ib_to_MN_inhib = 0*1e-2*msiemens # 5*1e-2*siemens # in siemens # Increase in inhibitory conductance in MN when receiving spike from Ib inhibitory interneuron
Ib_to_MN_synaptic_delay = 10*ms

### EQUATIONS BEING RUN BY BRIAN2 ##########################################################################
MN_equations = Equations('''
    dv/dt = (-I_leak - I_excit - I_inhib) / C_m : volt (unless refractory)
    I_leak = g_leak*(v - E_leak) : amp
    I_excit = clip(synaptic_input_excit + I_th, -inf*nA, 0*nA) : amp # Implementing the rheobase = the excitatory current is taken into account only if it is above the rheobase
    synaptic_input_excit = (ge*(v - E_excit)) : amp
    I_inhib = gi*(v - E_inhib) : amp
    dge/dt = ((input_weight * input_excit(t,i)) - ge) / tau : siemens
    dgi/dt = ((input_weight * input_inhib(t,i)) - gi) / tau : siemens
    tau : second
    g_leak : siemens
    C_m : farad
    refractory_period : second
    input_weight : 1
    I_th : amp
    ''')

# Check if there is indeed an exponential decay towards 0 (resting voltage)
RC_equations = Equations('''
    dv/dt = (input_RC(t,i)-v)/tau: volt (unless refractory)
    tau : second
    ''')

# muscle_fibers_equations = Equations('''
#     dforce/dt = ((fiber_force_rest - force) / force_decay_tau) - (I_force_rise / (1*farad)) : volt  # Force increase (twitch decrease) based on conductance
#     I_force_rise = g_force*(force - fiber_max_force) : amp
#     dg_force/dt = ((0*siemens) - g_force) / force_twitch_rise_tau : siemens
#     force_decay_tau : second
#     fiber_max_force : volt
#     force_twitch_rise_tau : second
#     ''')

muscle_fibers_equations = Equations('''
    dforce/dt = (-I_force_decay - I_force_rise) / (1*farad) : volt  # Force increase (twitch decrease) based on conductance
    I_force_decay = g_force_decay*(force - resting_force) : amp # correspond to leak current
    I_force_rise = g_force_rise*(force - fiber_max_force) : amp
    dg_force_rise/dt = ((0*siemens) - g_force_rise) / force_twitch_rise_tau : siemens # set at 0 because conductance stays at zero unlss there is presynaptic input
    g_force_decay : siemens 
    force_twitch_rise_tau : second
    resting_force : volt
    fiber_max_force : volt
    ''')

total_muscle_force_equations = Equations('''
    force_total_relative = ((force_total/volt) / muscle_maximal_force) * 100: 1
    force_total : volt
    ''')

# Check if there is indeed an exponential decay towards 0 (resting voltage)
Ib_interneurons_equations = Equations('''
    dv/dt = (input_Ib(t,i)+force_induced_current-v)/tau: volt (unless refractory)
    force_induced_current : volt
    tau : second
    ''')

In [151]:
### CREATE NEW FOLDER
new_directory = sim_name
new_filename = 'parameters.txt'

# Create the directory if it doesn't exist
if not os.path.exists(new_directory):
    os.makedirs(new_directory)
else:
    directory_n = 0
    while os.path.exists(new_directory):
        directory_n = directory_n+1
        new_directory = str(sim_name + "_iter_" + str(directory_n))
        if not os.path.exists(new_directory):
            os.makedirs(new_directory)
            break
        if directory_n > 99: # prevent infinite loop
            break
save_file_path = os.path.join(new_directory, new_filename)


In [152]:
### SAVE PARAMETERS

# Write the variables to the file
with open(save_file_path, 'w') as file:
    file.write(f"General parameters -----\n")
    file.write(f" - Time -----\n")
    file.write(f"       fsamp: {fsamp}\n")
    file.write(f"       true_duration: {true_duration}\n")
    file.write(f"       window_beginning_ignore: {window_beginning_ignore}\n")
    file.write(f"       window_end_ignore: {window_end_ignore}\n")
    file.write(f"       ISI_threshold_for_discontinuity: {ISI_threshold_for_discontinuity}\n")
    file.write(f" - Neurons population -----\n")
    file.write(f"       nb_pools: {nb_pools}\n")
    file.write(f"       nb_motoneurons_per_pool: {nb_motoneurons_per_pool}\n")
    file.write(f"       min_soma_diameter: {min_soma_diameter} micrometers\n")
    file.write(f"       max_soma_diameter: {max_soma_diameter} micrometers\n")
    file.write(f"       size_distribution_exponent: {size_distribution_exponent}\n")
    file.write(f"       nb_Renshaw_cells_per_pool: {nb_Renshaw_cells_per_pool}\n")
    file.write(f"       total_nb_Ib_interneurons: {total_nb_Ib_interneurons}\n")
    file.write(f"   Factor_analysis_on_all_pools: {factor_analysis_on_all_pools}\n")

    file.write(f"\n")
    file.write(f"Input parameters -----\n")
    file.write(f"   Common exitatory input -----\n")
    file.write(f"       nb_excitatory_inputs_per_pool: {nb_excitatory_inputs_per_pool}\n")
    file.write(f"       excitatory_input_baseline: {excitatory_input_baseline}\n")
    file.write(f"       excitatory_inputs_std: {excitatory_inputs_std}\n")
    file.write(f"       low_pass_filter_of_excitatory_input: {low_pass_filter_of_excitatory_input}\n")
    file.write(f"       excitatory_input_source: {excitatory_input_source}\n")
    file.write(f"       excitatory_input_sourcefile_path: {excitatory_input_sourcefile_path}\n")
    file.write(f"       excitatory_input_sourcefile_filename: {excitatory_input_sourcefile_filename}\n")
    file.write(f"       set_arbitrary_correlation_between_excitatory_inputs: {set_arbitrary_correlation_between_excitatory_inputs}\n")
    file.write(f"       within_pool_excitatory_input_correlation: {within_pool_excitatory_input_correlation}\n")
    file.write(f"       between_pool_excitatory_input_correlation: {between_pool_excitatory_input_correlation}\n")
    file.write(f"       distribution_of_excitation: {distribution_of_excitation}\n")
    file.write(f"   Common inhibitory input -----\n")
    file.write(f"       nb_inhibitory_inputs_per_pool: {nb_inhibitory_inputs_per_pool}\n")
    file.write(f"       inhibitory_input_mean: {inhibitory_input_mean}\n")
    file.write(f"       inhibitory_input_std: {inhibitory_input_std}\n")
    file.write(f"       low_pass_filter_of_inhibitory_input: {low_pass_filter_of_inhibitory_input}\n")
    file.write(f"       distribution_of_inhibition: {distribution_of_inhibition}\n")
    file.write(f"       set_arbitrary_correlation_between_inhibitory_and_excitatory_inputs: {set_arbitrary_correlation_between_inhibitory_and_excitatory_inputs}\n")
    file.write(f"       within_pool_excitatory_to_inhibitory_input_correlation: {within_pool_excitatory_to_inhibitory_input_correlation}\n")
    file.write(f"       between_pool_excitatory_to_inhibitory_input_correlation: {between_pool_excitatory_to_inhibitory_input_correlation}\n")
    file.write(f"   Independent noise (motor neurons) -----\n")
    file.write(f"       low_pass_filter_of_MN_independent_noise: {low_pass_filter_of_MN_independent_noise}\n")
    file.write(f"       MN_independent_noise_excit_std: {MN_independent_noise_excit_std}\n")
    file.write(f"       MN_independent_noise_inhib_std: {MN_independent_noise_inhib_std}\n")
    file.write(f"       low_pass_filter_of_other_cell_types_independent_noise: {low_pass_filter_of_other_cell_types_independent_noise}\n")
    file.write(f"   Independent noise (Renshaw cells and Ib interneurons) -----\n")
    file.write(f"       RC_independent_noise_std: {RC_independent_noise_std}\n")
    file.write(f"       Ib_interneruons_noise_std: {Ib_interneruons_noise_std}\n")

    file.write(f"\n")
    file.write(f"Electrophysiological poperties -----\n")
    file.write(f"   Common to all neurons -----\n")
    file.write(f"       voltage_rest: {voltage_rest}\n")
    file.write(f"       voltage_thresh: {voltage_thresh}\n")
    file.write(f"   Motor neurons reversal/equilibrium potentials -----\n")
    file.write(f"       E_leak: {E_leak}\n")
    file.write(f"       E_excit: {E_excit}\n")
    file.write(f"       E_inhib: {E_inhib}\n")
    file.write(f"   Constant and exponents to define electrophysiological properties of motor neurons (Caillet 2022) -----\n")
    file.write(f"       tau_MN_constant: {tau_MN_constant}\n")
    file.write(f"       tau_MN_exponent: {tau_MN_exponent}\n")
    file.write(f"       resistance_constant: {resistance_constant}\n")
    file.write(f"       resistance_exponent: {resistance_exponent}\n")
    file.write(f"       membrane_conductance_scaling: {membrane_conductance_scaling}\n")
    file.write(f"       rheobase_constant: {rheobase_constant}\n")
    file.write(f"       rheobase_exponent: {rheobase_exponent}\n")
    file.write(f"       rheobase_scaling: {rheobase_scaling}\n")
    file.write(f"       capacitance_constant: {capacitance_constant}\n")
    file.write(f"       capacitance_exponent: {capacitance_exponent}\n")
    file.write(f"       AHP_duration_constant: {AHP_duration_constant}\n")
    file.write(f"       AHP_duration_exponent: {AHP_duration_exponent}\n")
    file.write(f"       refractory_period_as_AHP_fraction: {refractory_period_as_AHP_fraction}\n")
    file.write(f"   Renshaw cells and Ib inhibitory interneurons -----\n")
    file.write(f"       tau_Renshaw: {tau_Renshaw}\n")
    file.write(f"       refractory_period_RC: {refractory_period_RC}\n")
    file.write(f"       tau_Ib_interneurons: {tau_Ib_interneurons}\n")
    file.write(f"       refractory_period_Ib_interneurons: {refractory_period_Ib_interneurons}\n")

    file.write(f"\n")
    file.write(f"Muscle fibers mechanical poperties -----\n")
    file.write(f"       fiber_force_rest: {fiber_force_rest}\n")
    file.write(f"       force_twitch_rise_tau_baseline: {force_twitch_rise_tau_baseline}\n")
    file.write(f"       force_decay_tau_baseline: {force_decay_tau_baseline}\n")
    file.write(f"       force_baseline_conductance_scalar: {force_baseline_conductance_scalar}\n")
    file.write(f"       smallest_MU_tau_multiplier: {smallest_MU_tau_multiplier}\n")
    file.write(f"       largest_MU_tau_multiplier: {largest_MU_tau_multiplier}\n")
    file.write(f"       smallest_MU_max_force: {smallest_MU_max_force}\n")
    file.write(f"       largest_MU_max_force: {largest_MU_max_force}\n")
    file.write(f"       axonal_conduction_velocity_constant: {axonal_conduction_velocity_constant}\n")
    file.write(f"       axonal_conduction_velocity_exponent: {axonal_conduction_velocity_exponent}\n")

    file.write(f"\n")
    file.write(f"Connectivity (synaptic parameters) -----\n")
    file.write(f"   Connectivity -----\n")
    file.write(f"       MN_to_Renshaw_connectivity_probability_within_pool: {MN_to_Renshaw_connectivity_probability_within_pool}\n")
    file.write(f"       Renshaw_to_MN_connectivity_probability_within_pool: {Renshaw_to_MN_connectivity_probability_within_pool}\n")
    file.write(f"       MN_to_Renshaw_connectivity_probability_across_pool: {MN_to_Renshaw_connectivity_probability_across_pool}\n")
    file.write(f"       Renshaw_to_MN_connectivity_probability_across_pool: {Renshaw_to_MN_connectivity_probability_across_pool}\n")
    file.write(f"       MN_to_Renshaw_connectivity_prevent_heteronymous_pool_overlap: {MN_to_Renshaw_connectivity_prevent_heteronymous_pool_overlap}\n")
    file.write(f"       Renshaw_to_MN_connectivity_prevent_heteronymous_pool_overlap: {Renshaw_to_MN_connectivity_prevent_heteronymous_pool_overlap}\n")
    file.write(f"       Ib_inh_int_to_MN_connectivity_probability: {Ib_inh_int_to_MN_connectivity_probability}\n")
    file.write(f"       Ib_inh_int_to_MN_connectivity_gaussian_nonbinary_weights: {Ib_inh_int_to_MN_connectivity_gaussian_nonbinary_weights}\n")
    file.write(f"       Ib_inh_int_to_MN_connectivity_probability_std: {Ib_inh_int_to_MN_connectivity_probability_std}\n")
    file.write(f"   Post-synaptic effects -----\n")
    file.write(f"       MN_to_Renshaw_excit: {MN_to_Renshaw_excit}\n")
    file.write(f"       Renshaw_to_MN_inhib: {Renshaw_to_MN_inhib}\n")
    file.write(f"       MN_RC_synpatic_delay: {MN_RC_synpatic_delay}\n")
    file.write(f"       MN_to_muscle_fiber_force: {MN_to_muscle_fiber_force}\n")
    file.write(f"       force_to_Ib_excit: {force_to_Ib_excit}\n")
    file.write(f"       Ib_to_MN_inhib: {Ib_to_MN_inhib}\n")
    file.write(f"       Ib_to_MN_synaptic_delay: {Ib_to_MN_synaptic_delay}\n")
    
    

In [153]:
# Define lerp (linear interpolation) function:
def lerp(a, b, t):
    return a + t * (b - a)

In [ ]:
####### GENERATE MOTOR NEURONS

motoneuron_soma_diameters = np.zeros(total_nb_motoneurons)
motoneuron_normalized_soma_diameters = np.zeros(total_nb_motoneurons)
for pooli in range(nb_pools):
    mni_offset = pooli*nb_motoneurons_per_pool
    for mni in range(nb_motoneurons_per_pool):
        motoneuron_soma_diameters[mni_offset+mni] = lerp(min_soma_diameter, max_soma_diameter, (mni/(nb_motoneurons_per_pool-1))**size_distribution_exponent )
        motoneuron_normalized_soma_diameters[mni_offset+mni] =lerp(0, 1, (mni/(nb_motoneurons_per_pool-1))**size_distribution_exponent )


# Plot histogram of motoneuron sizes
plt.figure()
# Create the histogram
counts, bins, patches = plt.hist(motoneuron_soma_diameters, density=True)
# Multiply the counts by 100 to convert to percentage
counts_percentage = counts * 100
# Plot the histogram again with the adjusted counts
plt.clf()  # Clear the current plot
plt.hist(motoneuron_soma_diameters, density=False, weights=np.ones_like(motoneuron_soma_diameters) * (100 / len(motoneuron_soma_diameters)),
         edgecolor='white', color='gray', alpha=1)
plt.vlines(min_soma_diameter,plt.ylim()[0],plt.ylim()[1],color='C1', label='Min soma diameter', linewidth=2)
plt.vlines(max_soma_diameter,plt.ylim()[0],plt.ylim()[1],color='C3', label='Max soma diameter', linewidth=2)
plt.legend()
plt.xlabel("Motoneuron size (soma diameter in micrometer)")
plt.ylabel("Proportion (% of total nb of motoneurons)")
plt.title("Distribution of motor neuron sizes")
new_filename = f'MN_sizes_distribution.png'
save_file_path = os.path.join(new_directory, new_filename)
plt.savefig(save_file_path)
plt.show()

plt.figure()
# Create the histogram
counts, bins, patches = plt.hist(motoneuron_normalized_soma_diameters, density=True)
# Multiply the counts by 100 to convert to percentage
counts_percentage = counts * 100
# Plot the histogram again with the adjusted counts
plt.clf()  # Clear the current plot
plt.hist(motoneuron_normalized_soma_diameters, density=False, weights=np.ones_like(motoneuron_normalized_soma_diameters) * (100 / len(motoneuron_normalized_soma_diameters)),
         edgecolor='white', color='gray', alpha=0.5)
plt.vlines(0,plt.ylim()[0],plt.ylim()[1],color='C1', label='Min soma diameter', linewidth=2)
plt.vlines(1,plt.ylim()[0],plt.ylim()[1],color='C3', label='Max soma diameter', linewidth=2)
plt.legend()
plt.xlabel("Normalized motoneuron size")
plt.ylabel("Proportion (% of total nb of motoneurons)")

plt.figure()
plt.plot(motoneuron_soma_diameters, color='gray')
plt.xlabel("Motoneuron idx")
plt.ylabel("MN size (soma diameter in micrometers)")
plt.title(f"Size of MNs according to index \n Mean soma diameter =  micrometers")
new_filename = f'MN_sizes_according_to_idx.png'
save_file_path = os.path.join(new_directory, new_filename)
plt.savefig(save_file_path)
plt.show()

In [ ]:
### Motor neurons electrophysiological properties

motoneuron_resistances = np.zeros(total_nb_motoneurons)
motoneuron_input_weights = np.zeros(total_nb_motoneurons)
motoneuron_capacitances = np.zeros(total_nb_motoneurons)
motoneurons_membrane_conductance = np.zeros(total_nb_motoneurons)
motoneurons_AHP_durations = np.zeros(total_nb_motoneurons)
motoneurons_refractory_periods = np.zeros(total_nb_motoneurons)
motoneurons_rheobases = np.zeros(total_nb_motoneurons)
motoneurons_taus = np.zeros(total_nb_motoneurons) # derived from experimental data regression (Caillet et al. 2022)
motoneurons_theoritical_taus = np.zeros(total_nb_motoneurons) # simply resistance*capacitance
for mni in range(total_nb_motoneurons):
    motoneuron_resistances[mni] = resistance_constant*(motoneuron_soma_diameters[mni]**resistance_exponent)
    motoneuron_input_weights[mni] = motoneuron_resistances[mni] / motoneuron_resistances[0] # normalized value so that smallest MN has weight of 1
    motoneuron_capacitances[mni] = capacitance_constant*(motoneuron_soma_diameters[mni]**capacitance_exponent)
    motoneurons_membrane_conductance[mni] = 1/motoneuron_resistances[mni] * membrane_conductance_scaling
    motoneurons_AHP_durations[mni] = AHP_duration_constant*(motoneuron_soma_diameters[mni]**AHP_duration_exponent)
    motoneurons_refractory_periods[mni] = motoneurons_AHP_durations[mni] * refractory_period_as_AHP_fraction
    motoneurons_rheobases[mni] = rheobase_constant*(motoneuron_soma_diameters[mni]**rheobase_exponent)
    motoneurons_taus[mni] = tau_MN_constant*(motoneuron_soma_diameters[mni]**(-1*tau_MN_exponent))
    motoneurons_theoritical_taus[mni] = motoneuron_resistances[mni]*motoneuron_capacitances[mni]*1e-2 # 1e-2 to get values within the 20 (largest MN) - 45 (smallest MN) ms range
plt.figure(figsize=(7,5))
fig, ax1 = plt.subplots()
curve1, = ax1.plot(motoneuron_resistances, label = "Resistance (ohms)", color = 'C1', linewidth = 3)
ax2 = ax1.twinx()
curve2, = ax2.plot(motoneuron_input_weights, label = "Input weight (normalized input resistance)", color = 'C5', linewidth = 2, linestyle=":")
curves = [curve1, curve2]
labels = [curve.get_label() for curve in curves]
ax1.legend(curves, labels, loc='best')
ax1.tick_params(axis='y', labelcolor='C1')
ax1.set_ylabel("Resistance (ohms)", color ='C1')
ax2.tick_params(axis='y', labelcolor='C5')
ax2.set_ylabel("Normalized input resistance (weight between 0 and 1)", color ='C5')
plt.figure(figsize=(7,5))
plt.plot(motoneuron_capacitances, label = "Capacitance (microFarads)", color = 'C2')
plt.legend()
plt.xlabel("MN index")
plt.ylabel("Capacitance (Farads)")
plt.figure(figsize=(7,5))
plt.plot(motoneurons_taus, label = "Membrane time constant", color = 'C9')
plt.plot(motoneurons_theoritical_taus, label = "Theoritical time constant (R*C)", linestyle="--", color = 'C9')
plt.legend()
plt.xlabel("MN index")
plt.ylabel("Tau (ms)")
plt.figure(figsize=(7,5))
plt.plot(motoneurons_membrane_conductance, label = "Membrane conductance (milliSiemens)", color = 'C3')
plt.legend()
plt.xlabel("MN index")
plt.ylabel("Membrane conductance (milliSiemens)")
plt.figure(figsize=(7,5))
plt.plot(motoneurons_refractory_periods, label = "Scaled refractory period (ms) as surrogate for the AHP duration", color = 'C6')
plt.legend()
plt.xlabel("MN index")
plt.ylabel("Refractory period (ms)")
plt.figure(figsize=(7,5))
plt.plot(motoneurons_rheobases, label = "Rheobase (nanoAmperes)", color = 'C7')
plt.legend()
plt.xlabel("MN index")
plt.ylabel("Rheobase (nanoAmperes)")

In [ ]:
### Motor unit force/mechanical properties

muscle_fibers_max_force = np.zeros(total_nb_motoneurons)
muscle_fibers_tau_multiplier = np.zeros(total_nb_motoneurons)
muscle_fibers_twitch_rise_taus = np.zeros(total_nb_motoneurons)
muscle_fibers_twitch_decay_taus = np.zeros(total_nb_motoneurons)
muscle_fibers_electromechanical_delay = np.zeros(total_nb_motoneurons)
for mni in range(total_nb_motoneurons):
    muscle_fibers_max_force[mni] = lerp(smallest_MU_max_force, largest_MU_max_force, motoneuron_normalized_soma_diameters[mni])
    muscle_fibers_tau_multiplier[mni] = lerp(smallest_MU_tau_multiplier, largest_MU_tau_multiplier, motoneuron_normalized_soma_diameters[mni])
    muscle_fibers_twitch_rise_taus[mni] = force_twitch_rise_tau_baseline * muscle_fibers_tau_multiplier[mni]
    muscle_fibers_twitch_decay_taus[mni] = force_decay_tau_baseline * muscle_fibers_tau_multiplier[mni]
    muscle_fibers_electromechanical_delay[mni] = (1/(axonal_conduction_velocity_constant*(motoneuron_soma_diameters[mni]**(axonal_conduction_velocity_exponent))))*second
muscle_maximal_force = np.sum(muscle_fibers_max_force)

plt.figure(figsize=(7,5))
plt.plot(muscle_fibers_max_force, label = "Motor unit torque (mN/m)", color = 'red')
plt.legend()
plt.xlabel("MN index")
plt.ylabel("Associated muscle fibers max (tetanus) torque (mN/m)")

plot_cumulative_fiber_force = False # Takes quite some time if set to true
if plot_cumulative_fiber_force:
    plt.figure(figsize=(5,8))
    bottom_barplot = np.zeros(nb_motoneurons_per_pool)
    mn_iter_i = -1
    for pooli in range(nb_pools):
        colormap_temp = cm.get_cmap(pool_cmap[pooli])
        for mni in range(nb_motoneurons_per_pool):
            mn_iter_i += 1
            if mni == np.round(nb_motoneurons_per_pool/2):
                plt.bar(0, muscle_fibers_max_force[mn_iter_i]/muscle_maximal_force*100,
                        color = colormap_temp(mni/nb_motoneurons_per_pool),
                        bottom = bottom_barplot,
                        label = f"Muscle fibers from pool #{pooli}")
            else:
                plt.bar(0, muscle_fibers_max_force[mn_iter_i]/muscle_maximal_force*100,
                        color = colormap_temp(mni/nb_motoneurons_per_pool),
                        bottom = bottom_barplot)  
            bottom_barplot += muscle_fibers_max_force[mn_iter_i]/muscle_maximal_force*100
    plt.xticks([])
    plt.xlabel(" ")
    plt.legend()
    plt.ylabel("Cumulative muscle fibers max (tetanus) force, as % of MVC")

plt.figure(figsize=(7,5))
plt.plot(muscle_fibers_twitch_rise_taus, label = "Twitch (force rise) time constant", color = 'orange')
plt.plot(muscle_fibers_twitch_decay_taus, label = "Relaxation (force decay) time constant", color = 'purple', alpha = 0.5)
plt.plot(muscle_fibers_electromechanical_delay, label = "Electromechanical delay", color = 'green')
plt.legend()
plt.xlabel("MN index")
plt.ylabel("second")

In [157]:
# Define a low-pass filter
def butter_lowpass(cutoff, fs, order=5):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    return b, a
def lowpass_filter(data, cutoff, fs, order=5):
    b, a = butter_lowpass(cutoff, fs, order=order)
    y = filtfilt(b, a, data)
    return y

# Low-pass filter artifact removal
duration_to_remove = 1/window_beginning_ignore # in second
Wind_s = duration_to_remove * 2
artifact_removal_window = windows.hann(round(fsamp * Wind_s))
artifact_removal_window = artifact_removal_window[:int(np.round(len(artifact_removal_window)/2))]
nb_samples_artifact_removal_window = len(artifact_removal_window)

In [158]:
# When rounding 2.7 for example, 70% to get 3 and 30% chance to get 2
def probabilistic_round(number):
    lower = int(number)  # The lower integer
    upper = lower + 1    # The upper integer
    decimal_part = number - lower
    
    return upper if random.random() < decimal_part else lower

In [159]:
# Generate random signal function
def Generate_filtered_gaussian_noise_input(low_pass_filter_cutoff, input_mean, scaling_std):
    # Generate random input
    temp_input = np.random.normal(0, 1, int(duration_with_ignored_window * fsamp))
    # Apply artifact removal window to the end of the signal
    end_ignore_start = int(duration_with_ignored_window * fsamp) - int(np.round(window_end_ignore * fsamp))
    temp_input[end_ignore_start:] = temp_input[end_ignore_start:] * np.flip(artifact_removal_window)
    # Apply artifact removal window to the beginning of the signal
    beginning_ignore_end = int(np.round(window_beginning_ignore * fsamp))
    temp_input[:beginning_ignore_end] = temp_input[:beginning_ignore_end] * artifact_removal_window
    # Apply low-pass filter
    temp_input = lowpass_filter(temp_input, low_pass_filter_cutoff, fsamp)
    # Normalize the signal
    temp_input = temp_input - np.mean(temp_input)
    temp_input = temp_input / np.std(temp_input)
    # Scale and add mean
    temp_input = temp_input * scaling_std
    temp_input = temp_input + input_mean
    
    return temp_input

In [160]:
# Define the softmax function, to randomly distribute the input signals into pools of MNs, while making sure that the sum of the weights is 1.
def softmax_with_temperature(logits, temperature=1.0):
    """
    Compute the softmax of a list of logits with a temperature parameter.

    Parameters:
    logits (list or numpy array): The input logits.
    temperature (float): The temperature parameter.

    Returns:
    numpy array: The softmax probabilities.
    """
    # Convert logits to numpy array if they are not already
    logits = np.array(logits)
    
    # Apply the temperature parameter
    logits = logits / temperature
    
    # Compute the exponentials of the scaled logits
    exp_logits = np.exp(logits - np.max(logits))  # Subtract max for numerical stability
    
    # Compute the softmax probabilities
    softmax_probs = exp_logits / np.sum(exp_logits)
    
    return softmax_probs

In [161]:
# Generate random signal function
def Generate_filtered_burst_input(burst_frequency, input_peak, burst_sharpness):
    # Generate gradually increasing input
    temp_input = np.linspace(0, duration_with_ignored_window/second, int(fsamp*duration_with_ignored_window/second))
    # Generate burst with powers of sin
    temp_input = np.abs(np.sin(temp_input*np.pi*burst_frequency)**burst_sharpness)
    # Apply artifact removal window to the end of the signal
    end_ignore_start = int(duration_with_ignored_window * fsamp) - int(np.round(window_end_ignore * fsamp))
    temp_input[end_ignore_start:] = temp_input[end_ignore_start:] * np.flip(artifact_removal_window)
    # Apply artifact removal window to the beginning of the signal
    beginning_ignore_end = int(np.round(window_beginning_ignore * fsamp))
    temp_input[:beginning_ignore_end] = temp_input[:beginning_ignore_end] * artifact_removal_window
    # Scale and add mean
    temp_input = temp_input * input_peak
    
    return temp_input

In [162]:
### FUNCTIONS TO MODIFY THE INPUTS IN ORDER TO GET THE DESIRED PAIRWISE CORREL COEF BETWEEN INPUTS (if the option has been set to true)

def is_positive_definite(X):
    try:
        np.linalg.cholesky(X)
        return True
    except np.linalg.LinAlgError:
        return False

def nearest_positive_definite(A):
    B = (A + A.T) / 2
    U, s, Vt = np.linalg.svd(B)
    H = np.dot(Vt.T * s, Vt)
    A2 = (B + H) / 2
    A3 = (A2 + A2.T) / 2
    if is_positive_definite(A3):
        return A3
    spacing = np.spacing(np.linalg.norm(A))
    I = np.eye(A.shape[0])
    k = 1
    while not is_positive_definite(A3):
        A3 += I * spacing * k
        k += 1
    return A3

def adjust_correlation(data, R_desired, threshold_for_adjusting_correl=0.05):
    """
    Adjust correlation of 'data' so its correlation matrix approximates R_desired.
    Handles negative correlations by a post-processing sign-flip heuristic.
    """
    data = np.asarray(data, dtype=float)
    n_samples, n_vars = data.shape

    if R_desired.shape != (n_vars, n_vars):
        raise ValueError("R_desired must be an n_vars x n_vars matrix.")
    if not np.allclose(R_desired, R_desired.T):
        raise ValueError("R_desired must be symmetric.")
    if not np.all(np.diag(R_desired) == 1.0):
        raise ValueError("Diagonal elements of R_desired must be 1.")

    # Store the sign pattern of desired correlations
    sign_pattern = np.sign(R_desired)

    # Use absolute values to form a positive definite target
    R_abs = np.abs(R_desired)

    # Make sure R_abs is positive definite
    R_abs_pd = R_abs if is_positive_definite(R_abs) else nearest_positive_definite(R_abs)

    # Standardize data
    orig_means = np.mean(data, axis=0)
    orig_stds = np.std(data, axis=0, ddof=1)
    if np.any(orig_stds == 0):
        raise ValueError("One or more input variables are constant; cannot adjust correlation.")
    data_std = (data - orig_means) / orig_stds

    # Compute current correlation
    R_current = np.corrcoef(data_std, rowvar=False)
    # Make sure R_current is PD
    R_current_pd = R_current if is_positive_definite(R_current) else nearest_positive_definite(R_current)

    # Cholesky decompositions
    L_current = np.linalg.cholesky(R_current_pd)
    L_desired = np.linalg.cholesky(R_abs_pd)

    # Whiten data
    inv_L_current = np.linalg.inv(L_current)
    data_white = data_std @ inv_L_current

    # Apply desired correlation (absolute values)
    data_transformed_std = data_white @ L_desired

    # Rescale back
    data_transformed = data_transformed_std * orig_stds + orig_means

    # Now, adjust signs if needed
    # Check the resulting correlations
    R_final = np.corrcoef(data_transformed, rowvar=False)

    # Check the resulting correlation and impose correlation relative to the first (first column of data) input if necessary
    for inputi in range(len(data_transformed[0])):
        if inputi==0:
            continue # skipping the first input (correlation with itself)
        else:
            temp_correl = np.corrcoef(data_transformed[:,0],data_transformed[:,inputi])[0, 1]
            temp_desired_correl = R_desired[0, inputi]
            diff_actual_VS_desired = temp_correl - temp_desired_correl # negative if the correlation is too low, positive if the correlation is too high
            if (abs(diff_actual_VS_desired) > threshold_for_adjusting_correl) and (diff_actual_VS_desired < 0): # if the correlation is too low
                data_transformed[:, inputi] = data_transformed[:,inputi]*(1-abs(diff_actual_VS_desired)) + (data_transformed[:,0]*abs(diff_actual_VS_desired))
            if (abs(diff_actual_VS_desired) > threshold_for_adjusting_correl) and (diff_actual_VS_desired > 0): # if the correlation is too high
                data_transformed[:, inputi] = data_transformed[:,inputi]*(1-abs(diff_actual_VS_desired)) - (data_transformed[:,0]*abs(diff_actual_VS_desired))

    # If a desired correlation is negative but we got a positive one, flip one column
    # This is a heuristic. We try flipping the second column in the pair.
    min_R_final_for_flip = 0.2  # adjust this threshold as needed
    min_R_desired_for_flip = -0.2  # adjust this threshold as needed
    for i in range(n_vars):
        for j in range(i+1, n_vars):
            if R_desired[i, j] < min_R_desired_for_flip:  # desired negative correlation exceeding a threshold
                if R_final[i, j] > min_R_final_for_flip:  # got a positive correlation exceeding a threshold instead
                    # Flip the sign of column j
                    data_transformed[:, j] = -data_transformed[:, j]
                    # Recalculate R_final after the flip
                    # R_final = np.corrcoef(data_transformed, rowvar=False)

    return data_transformed

# Example usage:
# A = np.random.randn(1000, 3)
# R_desired = np.array([[1.0, -0.5, 0.3],
#                       [-0.5, 1.0, -0.2],
#                       [0.3, -0.2, 1.0]])
# data_new = adjust_correlation(A, R_desired)
# np.corrcoef(data_new, rowvar=False) should reflect the sign pattern of R_desired.



In [163]:
# Fetch, resample, and standardize input from experimental data (PCs)

def High_pass_filter_loaded_data(signal, cutoff, fs, order=2):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='high', analog=False)
    y = filtfilt(b, a, signal)
    return y


def Load_experimental_data_as_input(mat_file, PC_index):
    input_sample_size = size(mat_file['PCA_components'],0) # number of samples
    num_samples_for_resampling = int(input_sample_size * fsamp / excitatory_input_sourcefile_fsamp)
    # Get signal
    temp_signal = mat_file['PCA_components'][:,PC_index]
    # temp_signal = High_pass_filter_loaded_data(temp_signal, 0.3, excitatory_input_sourcefile_fsamp)
    # resample to appropriate fsamp
    resampled_signal_temp = resample(temp_signal, num_samples_for_resampling)
    # cut signal to be of the appropriate duration
    duration_of_sim_in_samples = int(fsamp*duration_with_ignored_window/second)
    if duration_of_sim_in_samples > len(resampled_signal_temp):
        raise ValueError(f"Error: your simulation duration ({duration_with_ignored_window}) is longer than the duration of the input signal ({len(resampled_signal_temp)/fsamp} s).")
    resampled_signal_temp = resampled_signal_temp[0:int(fsamp*duration_with_ignored_window/second)]
    # Standardize the signal (mean-center, std of 1)
    resampled_signal_temp = (resampled_signal_temp - np.mean(resampled_signal_temp)) / np.std(resampled_signal_temp)

    return resampled_signal_temp

In [ ]:
### MN EXCITATORY INPUT - Fetch (or generate) and distribute inputs

total_nb_of_excitatory_inputs = nb_pools * nb_excitatory_inputs_per_pool
MN_excit_input = {}
# Create or fetch excitatory inputs
if excitatory_input_source == 'load_experimental_data':
    mat_file_imported = False
input_iter_i = -1
for pooli in range(nb_pools):
    MN_excit_input[pooli] = {}
    for inputi in range(nb_excitatory_inputs_per_pool):
        input_iter_i += 1
        MN_excit_input[pooli][inputi] = []
        if excitatory_input_source == 'generate_synthetic_input':
            temp_input = Generate_filtered_gaussian_noise_input(low_pass_filter_of_excitatory_input, 0, 1)
        elif excitatory_input_source == 'load_experimental_data':
            if not mat_file_imported:
                print("Loading experimental data...")
                mat_file_imported = True
                mat_file = scipy.io.loadmat(str(excitatory_input_sourcefile_path+excitatory_input_sourcefile_filename))
                # Access variables: data = mat_file['variable_name']
            print(f"   ...loading PC{input_iter_i} as input for pool {pooli}, input {inputi}")
            temp_input = Load_experimental_data_as_input(mat_file,input_iter_i)
        elif excitatory_input_source == 'generate_burst_input':
            temp_input = Generate_filtered_burst_input(2, excitatory_inputs_std*50, 1000) # 2 bursts each second, with amplitude 5 (in millisiemens), with shaprness (how thin the bursts are) 100
        else:
            raise ValueError("Please select a valid excitatory input source: 'generate_synthetic_input' or 'load_experimental_data'")
        MN_excit_input[pooli][inputi].append(temp_input)

    
if set_arbitrary_correlation_between_excitatory_inputs:
    all_data_list = []
    for pooli in range(nb_pools):
        for inputi in range(nb_excitatory_inputs_per_pool):
            temp_input = MN_excit_input[pooli][inputi][0]  # if it's stored as a list with one element
            # Make sure temp_input is a 1D numpy array: shape (n_samples,)
            all_data_list.append(np.squeeze(temp_input))
    # Now stack them all into a 2D array: shape (total_nb_of_excitatory_inputs, n_samples)
    original_data_vectors = np.vstack(all_data_list)
    # Transpose so that shape is (n_samples, total_nb_of_excitatory_inputs)
    original_data_vectors = original_data_vectors.T

    total_nb_of_excitatory_inputs = nb_pools * nb_excitatory_inputs_per_pool
    target_correlation_matrix = np.zeros((total_nb_of_excitatory_inputs, total_nb_of_excitatory_inputs))
    for i in range(total_nb_of_excitatory_inputs):
        for j in range(total_nb_of_excitatory_inputs):
            pooli, inputi = divmod(i, nb_excitatory_inputs_per_pool)
            poolj, inputj = divmod(j, nb_excitatory_inputs_per_pool)
            if i == j:
                target_correlation_matrix[i, j] = 1.0
            elif pooli == poolj:
                target_correlation_matrix[i, j] = within_pool_excitatory_input_correlation
            else:
                target_correlation_matrix[i, j] = between_pool_excitatory_input_correlation


    transformed_data_vectors = adjust_correlation(original_data_vectors, target_correlation_matrix)

    # Distribute the transformed data vectors to the MN_excit_input dictionary
    iter_total = 0
    for pooli in range(nb_pools):
        for inputi in range(nb_excitatory_inputs_per_pool):
            MN_excit_input[pooli][inputi] = transformed_data_vectors[:, iter_total]
            iter_total += 1

# Stanardize the excitatory inputs to the desired STD amplitude
iter_total = 0
for pooli in range(nb_pools):
    for inputi in range(nb_excitatory_inputs_per_pool):
        temp_input = MN_excit_input[pooli][inputi]
        temp_input = (temp_input/np.std(temp_input))*excitatory_inputs_std
        MN_excit_input[pooli][inputi] = temp_input
        iter_total += 1

if set_same_excitatory_input_for_all_pools:
    for pooli in range(nb_pools):
        for inputi in range(nb_excitatory_inputs_per_pool):
            temp_input = MN_excit_input[0][0]
            MN_excit_input[pooli][inputi] = temp_input


excit_input_pairwise_correl = np.zeros((nb_excitatory_inputs_per_pool*nb_pools, nb_excitatory_inputs_per_pool*nb_pools))
iter_i = -1
for pooli in range(nb_pools):
    for inputi in range(nb_excitatory_inputs_per_pool):
        iter_i += 1
        iter_j = -1
        for poolj in range(nb_pools):
            for inputj in range(nb_excitatory_inputs_per_pool):
                iter_j += 1
                excit_input_pairwise_correl[iter_i,iter_j] = np.corrcoef(
                    MN_excit_input[pooli][inputi], MN_excit_input[poolj][inputj])[0, 1]


# Plot the excitatory input pairwise correlation matrix
custom_matrix_labels = []
for pooli in range(nb_pools):
    for inputi in range(nb_excitatory_inputs_per_pool):
        custom_matrix_labels.append(f'pool#{pooli}_input#{inputi}')
plt.figure(figsize=(8, 6))
sns.heatmap(excit_input_pairwise_correl, annot=True, cmap="RdYlBu_r", fmt=".2f", linewidths=0.5,
            xticklabels=custom_matrix_labels, yticklabels=custom_matrix_labels,
            vmin=-1, vmax=1)
plt.title(f"Pariwise correlation between excitatory inputs  \n Mean correl of inputs with first input (pool 0, input 0) = {np.mean(excit_input_pairwise_correl[0,1:]):.2f}")
new_filename = f'Pariwise_correlation_of_excitatory_input_signals.png'
save_file_path = os.path.join(new_directory, new_filename)
plt.savefig(save_file_path)
plt.show()

plt.figure(figsize=(15,5))
for pooli in range(nb_pools):
    colormap_temp = cm.get_cmap(pool_cmap[pooli])
    for inputi in range(nb_excitatory_inputs_per_pool):
        plt.plot(np.transpose(MN_excit_input[pooli][inputi]), alpha = 0.75, label=f'Excitatory input #{inputi}, pool#{pooli}',
                 color = colormap_temp(0.2+(inputi/(nb_excitatory_inputs_per_pool*2))))
plt.xlabel("Time (ms)")
plt.ylabel("Excitatory input variance (milliSiemens)")
plt.legend()
new_filename = f'Excitatory_input_signals.png'
save_file_path = os.path.join(new_directory, new_filename)
plt.savefig(save_file_path)
plt.show()


In [ ]:
### DISTRIBUTE THE EXCITATORY INPUTS TO THE POOLS
# N MN x N inputs matrices
excit_inputs_to_mn_weight_matrix = np.zeros((total_nb_motoneurons, total_nb_of_excitatory_inputs))
input_iter_i = -1
fig, ax = plt.subplots(1, nb_pools, figsize=(20,8))
if nb_pools == 1:  # Ensure ax is iterable for a single subplot
    ax = [ax]
for pooli in range(nb_pools):
    neurons_idx_of_current_pool = np.arange(nb_motoneurons_per_pool) + (pooli * nb_motoneurons_per_pool)
    inputs_idx_of_current_pool = np.arange(nb_excitatory_inputs_per_pool) + (pooli * nb_excitatory_inputs_per_pool)
    for inputi in range(nb_excitatory_inputs_per_pool):
        column_idx = inputs_idx_of_current_pool[inputi]
        if distribution_of_excitation == 'homogeneous_constant_sum_of_weights':
            excit_inputs_to_mn_weight_matrix[neurons_idx_of_current_pool, column_idx] = 1.0 / nb_excitatory_inputs_per_pool
        else:
            excit_inputs_to_mn_weight_matrix[neurons_idx_of_current_pool, column_idx] = (np.random.uniform(0,1,nb_motoneurons_per_pool) / nb_excitatory_inputs_per_pool)
    if distribution_of_excitation == 'heterogeneous_constant_sum_of_weights':
        for mni in neurons_idx_of_current_pool:
            excit_inputs_to_mn_weight_matrix[mni, inputs_idx_of_current_pool] = softmax_with_temperature(
                excit_inputs_to_mn_weight_matrix[mni, inputs_idx_of_current_pool], temperature=0.1)

            
    # Plotting
    x_plot_mns = range(nb_motoneurons_per_pool)
    bottom_barplot = np.zeros(nb_motoneurons_per_pool)
    colormap_temp = cm.get_cmap(pool_cmap[pooli])
    input_iter_i = -1
    for inputi in inputs_idx_of_current_pool:
        input_iter_i += 1
        ax[pooli].bar(x_plot_mns,
            excit_inputs_to_mn_weight_matrix[neurons_idx_of_current_pool, inputi],
            color = colormap_temp(0.2+(input_iter_i/(nb_excitatory_inputs_per_pool*2))),
            bottom = bottom_barplot,
            label = f'Pool #{pooli}, input #{inputi}')
        bottom_barplot += excit_inputs_to_mn_weight_matrix[neurons_idx_of_current_pool, inputi]
    ax[pooli].set_xlabel('Motoneurons')
    ax[pooli].set_ylabel('Proportion of common inputs')
    ax[pooli].set_title(f'Pool #{pooli}')
    ax[pooli].legend()
plt.tight_layout(rect=[0,0,1,0.96])
plt.suptitle("Distribution of common excitatory inputs")
new_filename = f'Excitatory_common_input_distribution_to_MN.png'
save_file_path = os.path.join(new_directory, new_filename)
plt.savefig(save_file_path)
plt.show()

plt.figure(figsize=(7,15))
sns.heatmap(excit_inputs_to_mn_weight_matrix, annot=False, cmap="RdYlBu_r", fmt=".2f", linewidths=0.5,
            xticklabels=custom_matrix_labels, vmin=0, vmax=1)
plt.title("Distribution of common excitatory inputs - matrix format")
plt.xlabel("Input #")
plt.ylabel("Motoneuron #")
new_filename = f'Excitatory_common_input_distribution_to_MN_matrix_format.png'
save_file_path = os.path.join(new_directory, new_filename)
plt.savefig(save_file_path)
plt.show()

In [166]:
def set_correlations_between_inhib_and_excit_inputs(A, B, R_desired):
    """
    Adjust the signals in A so that each column of A has the desired correlation 
    with each column of B, as specified by R_desired. The added orthogonal noise
    is low-pass filtered with a given cutoff frequency f_cut.

    Parameters
    ----------
    A : np.ndarray of shape (N, n_A)
        Data matrix to be modified.
    B : np.ndarray of shape (N, n_B)
        Reference data matrix (unchanged).
    R_desired : np.ndarray of shape (n_A, n_B)
        Desired correlation matrix between columns of A and columns of B.

    Returns
    -------
    A_new : np.ndarray of shape (N, n_A)
        Transformed A with the desired cross-correlation structure relative to B.
    """
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    N, n_A = A.shape
    N2, n_B = B.shape

    if N != N2:
        raise ValueError("A and B must have the same number of rows (samples).")

    if R_desired.shape != (n_A, n_B):
        raise ValueError("R_desired must be of shape (n_A, n_B).")

    # 1. Standardize A and B
    mu_A = A.mean(axis=0)
    sigma_A = A.std(axis=0, ddof=1)
    if np.any(sigma_A == 0):
        raise ValueError("Some columns of A are constant; cannot adjust correlation.")

    mu_B = B.mean(axis=0)
    sigma_B = B.std(axis=0, ddof=1)
    if np.any(sigma_B == 0):
        raise ValueError("Some columns of B are constant; cannot define correlation.")

    # standardize A and B
    A_std = (A - mu_A) / sigma_A
    B_std = (B - mu_B) / sigma_B

    A_new_std = np.zeros((N, n_A))

    # each column of A can be modified only twice = once for the between pool correlation, once for the within-pool correlation
    for col_A in range(n_A):
        intra_pool_correl_change_happened_to_col_A = False
        inter_pool_correl_change_happened_to_col_A = False
        A_i_std = A_std[:, col_A]
        pool_of_A = col_A % nb_inhibitory_inputs_per_pool
        for col_B in range(n_B):
            pool_of_B = col_B % nb_excitatory_inputs_per_pool
            # Transformation has already been applied
            if intra_pool_correl_change_happened_to_col_A and inter_pool_correl_change_happened_to_col_A:
                continue

            target_correl = R_desired[col_A, col_B]
            current_correl = np.corrcoef(A_i_std, B_std[:, col_B])[0, 1]
            correl_diff_to_desired = target_correl - current_correl
            # if correl_diff_to_desired > 0, corelation must be increased
            # if correl_diff_to_desired < 0, corelation must be decreased
            # if target_correl < 0 & correl_diff_to_desired < 0, correlation decrease must happen through anti correlation
            # if target_correl > 0 & current_diff > 0, correlation increase must happen through correlation
            # in other cases, change in correlation should happen by adding noise (bring the correlation closer to 0)
            if (target_correl < 0 and correl_diff_to_desired < 0) or (target_correl > 0 and correl_diff_to_desired > 0):
                A_i_std = (A_i_std * (1 - np.abs(correl_diff_to_desired))) + (correl_diff_to_desired * B_std[:, col_B])
            else:
                A_i_std = (A_i_std * (1 - np.abs(correl_diff_to_desired))) + (Generate_filtered_gaussian_noise_input(
                    low_pass_filter_of_inhibitory_input,0,1) * np.abs(correl_diff_to_desired))
                
            # Apply the transformation
            A_new_std[:, col_A] = A_i_std

            # Set the boolean variables to True if the transformation has been applied
            if (pool_of_A == pool_of_B):
                intra_pool_correl_change_happened_to_col_A = True
            if (pool_of_A != pool_of_B):
                inter_pool_correl_change_happened_to_col_A = True

    # # check results
    # results_corr = np.corrcoef(A_new_std, B_std, rowvar=False)[0:n_A, n_A:n_A+n_B] # should be close to R_desired.
    # plt.subplot(2, 1, 1)
    # sns.heatmap(results_corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
    # plt.title('Correlation matrix')
    # plt.subplot(2, 1, 2)
    # sns.heatmap(R_desired, annot=True, cmap='coolwarm', vmin=-1, vmax=1)
    # plt.title('Desired correlation matrix')

    # Rescale to original scale
    A_new = A_new_std * sigma_A + mu_A
    return A_new

# Example usage:
# A = np.random.randn(1000, 6)
# B = np.random.randn(1000, 3)
# R_desired = np.array([[0.5, 0.2, -0.3],
#                       [0.0, 0.5, 0.5],
#                       [0.4, -0.1, 0.0],
#                       [0.3, 0.3, 0.3],
#                       [0.2, 0.2, -0.2],
#                       [-0.1, 0.4, 0.4]])
# A_new = set_correlations_between_inhib_and_excit_inputs(A, B, R_desired)
# # Check correlations:
# # np.corrcoef(A_new, B, rowvar=False)[0:n_A, n_A:n_A+n_B] should be close to R_desired.


In [ ]:
### MN INHIBITORY INPUT - Fetch (or generate) and distribute inputs

total_nb_of_inhibitory_inputs = nb_pools * nb_inhibitory_inputs_per_pool
MN_inhib_input = {}
# Create or fetch inhibitory inputs
for pooli in range(nb_pools):
    MN_inhib_input[pooli] = {}
    for inputi in range(nb_inhibitory_inputs_per_pool):
        MN_inhib_input[pooli][inputi] = []
        temp_input = Generate_filtered_gaussian_noise_input(low_pass_filter_of_inhibitory_input, 0, 1)
        MN_inhib_input[pooli][inputi].append(temp_input)
        MN_inhib_input[pooli][inputi] = MN_inhib_input[pooli][inputi][0]

if set_arbitrary_correlation_between_inhibitory_and_excitatory_inputs:
    # get inhibitory input (to be transformed) #######
    all_data_list = []
    for pooli in range(nb_pools):
        for inputi in range(nb_inhibitory_inputs_per_pool):
            temp_input = MN_inhib_input[pooli][inputi]  # if it's stored as a list with one element
            # Make sure temp_input is a 1D numpy array: shape (n_samples,)
            all_data_list.append(np.squeeze(temp_input))
    # Now stack them all into a 2D array: shape (total_nb_of_excitatory_inputs, n_samples)
    matrix_of_inhib_data = np.vstack(all_data_list)
    # Transpose so that shape is (n_samples, total_nb_of_excitatory_inputs)
    matrix_of_inhib_data = matrix_of_inhib_data.T

    # get excitatory inputs (reference) #######
    all_data_list = []
    for pooli in range(nb_pools):
        for inputi in range(nb_excitatory_inputs_per_pool):
            temp_input = MN_excit_input[pooli][inputi]  # if it's stored as a list with one element
            # Make sure temp_input is a 1D numpy array: shape (n_samples,)
            all_data_list.append(np.squeeze(temp_input))
    # Now stack them all into a 2D array: shape (total_nb_of_excitatory_inputs, n_samples)
    matrix_of_excit_data = np.vstack(all_data_list)
    # Transpose so that shape is (n_samples, total_nb_of_excitatory_inputs)
    matrix_of_excit_data = matrix_of_excit_data.T

    # Create the target correlation matrix
    total_nb_of_excitatory_inputs = nb_pools * nb_excitatory_inputs_per_pool
    total_nb_of_inhibitory_inputs = nb_pools * nb_inhibitory_inputs_per_pool
    target_correlation_matrix = np.zeros((total_nb_of_excitatory_inputs, total_nb_of_inhibitory_inputs))
    for i in range(total_nb_of_excitatory_inputs):
        for j in range(total_nb_of_inhibitory_inputs):
            pooli, inputi = divmod(i, nb_excitatory_inputs_per_pool)
            poolj, inputj = divmod(j, nb_inhibitory_inputs_per_pool)
            if pooli == poolj:
                target_correlation_matrix[i, j] = within_pool_excitatory_to_inhibitory_input_correlation
            else:
                target_correlation_matrix[i, j] = between_pool_excitatory_to_inhibitory_input_correlation

    transformed_data_vectors = set_correlations_between_inhib_and_excit_inputs(matrix_of_inhib_data, matrix_of_excit_data, target_correlation_matrix.T)

    # Distribute the transformed data vectors to the MN_inhib_input dictionary
    iter_total = 0
    for pooli in range(nb_pools):
        for inputi in range(nb_inhibitory_inputs_per_pool):
            MN_inhib_input[pooli][inputi] = transformed_data_vectors[:, iter_total]
            iter_total += 1

# Compute pairwise correlation between excitatory and inhibitoy inputs
inhib_input_pairwise_correl = np.zeros((total_nb_of_excitatory_inputs, total_nb_of_inhibitory_inputs))
for input_exc in range(total_nb_of_excitatory_inputs):
    for input_inh in range(total_nb_of_inhibitory_inputs):
        inhib_input_pairwise_correl[input_exc, input_inh] = np.corrcoef(
            MN_excit_input[input_exc//nb_excitatory_inputs_per_pool][input_exc%nb_excitatory_inputs_per_pool],
            MN_inhib_input[input_inh//nb_inhibitory_inputs_per_pool][input_inh%nb_inhibitory_inputs_per_pool])[0,1]

# Standardize the inhibitory inputs to the desired STD amplitude and mean
iter_total = 0
for pooli in range(nb_pools):
    for inputi in range(nb_inhibitory_inputs_per_pool):
        temp_input = MN_inhib_input[pooli][inputi]
        temp_input = (temp_input/np.std(temp_input))*inhibitory_input_std
        MN_inhib_input[pooli][inputi] = temp_input
        iter_total += 1

# Plot the inhibitory inputs pairwise correlation matrix
custom_matrix_labels_Y = []
custom_matrix_labels_X = []
for pooli in range(nb_pools):
    for input_inh in range(nb_inhibitory_inputs_per_pool):
        custom_matrix_labels_Y.append(f'pool#{pooli}_input#{input_inh}')
    for input_exc in range(nb_excitatory_inputs_per_pool):
        custom_matrix_labels_X.append(f'pool#{pooli}_input#{input_exc}')
plt.figure(figsize=(8, 6))
sns.heatmap(inhib_input_pairwise_correl.T, annot=True, cmap="PiYG", fmt=".2f", linewidths=0.5,
            xticklabels=custom_matrix_labels_X, yticklabels=custom_matrix_labels_Y,
            vmin=-1, vmax=1)
plt.title("Correlation between inhibitory inputs and the excitatory inputs \n")
plt.xlabel("Excitatory input")
plt.ylabel("Inhibitory input")
new_filename = f'correlation_of_inhibtory_input_with_excitatory_inputs.png'
save_file_path = os.path.join(new_directory, new_filename)
plt.savefig(save_file_path)
plt.show()

plt.figure(figsize=(15,5))
for pooli in range(nb_pools):
    colormap_temp = cm.get_cmap(pool_cmap[pooli])
    for inputi in range(nb_inhibitory_inputs_per_pool):
        plt.plot(np.transpose(MN_inhib_input[pooli][inputi]), alpha = 0.75, label=f'Inhibbitory input #{inputi}, pool#{pooli}',
                 color = colormap_temp(0.8-(inputi/(nb_inhibitory_inputs_per_pool*3))))
plt.xlabel("Time (ms)")
plt.ylabel("Inhibitory input (milliSiemens)")
plt.legend()
new_filename = f'Inhibitory_input_signals.png'
save_file_path = os.path.join(new_directory, new_filename)
plt.savefig(save_file_path)
plt.show()



In [ ]:

### DISTRIBUTE THE INHIBITORY INPUTS TO THE POOLS
# N MN x N inputs matrices
inhib_inputs_to_mn_weight_matrix = np.zeros((total_nb_motoneurons, total_nb_of_inhibitory_inputs))
input_iter_i = -1
fig, ax = plt.subplots(1, nb_pools, figsize=(20,8))
if nb_pools == 1:  # Ensure ax is iterable for a single subplot
    ax = [ax]
for pooli in range(nb_pools):
    if nb_inhibitory_inputs_per_pool < 1:
        continue
    neurons_idx_of_current_pool = np.arange(pooli*nb_motoneurons_per_pool, (pooli+1)*nb_motoneurons_per_pool)
    inputs_idx_of_current_pool = np.arange(pooli*nb_inhibitory_inputs_per_pool, (pooli+1)*nb_inhibitory_inputs_per_pool)
    for inputi in range(nb_inhibitory_inputs_per_pool):
        input_iter_i += 1
        if distribution_of_inhibition == 'homogeneous_constant_sum_of_weights':
            inhib_inputs_to_mn_weight_matrix[neurons_idx_of_current_pool, input_iter_i] = 1/nb_inhibitory_inputs_per_pool
        else: # if distribution_of_inhibition == 'heterogeneous_random_sum_of_weights' or 'heterogeneous_constant_sum_of_weights'
            inhib_inputs_to_mn_weight_matrix[neurons_idx_of_current_pool, input_iter_i] = np.random.uniform(0,1,nb_motoneurons_per_pool)*2/nb_inhibitory_inputs_per_pool # *2 to get a mean of 1
    if distribution_of_inhibition == 'heterogeneous_constant_sum_of_weights':
        for mni in neurons_idx_of_current_pool:
            inhib_inputs_to_mn_weight_matrix[mni, inputs_idx_of_current_pool] = softmax_with_temperature(
                inhib_inputs_to_mn_weight_matrix[mni, inputs_idx_of_current_pool], temperature=0.1)
            
    # Plotting
    x_plot_mns = range(nb_motoneurons_per_pool)
    bottom_barplot = np.zeros(nb_motoneurons_per_pool)
    colormap_temp = cm.get_cmap(pool_cmap[pooli])
    for inputi in inputs_idx_of_current_pool:
        ax[pooli].bar(x_plot_mns,
            inhib_inputs_to_mn_weight_matrix[neurons_idx_of_current_pool, inputi],
            color = colormap_temp(0.8-(inputi/(nb_inhibitory_inputs_per_pool*3))),
            bottom = bottom_barplot,
            label = f'Pool #{pooli}, input #{inputi}')
        bottom_barplot += inhib_inputs_to_mn_weight_matrix[neurons_idx_of_current_pool, inputi]
    ax[pooli].set_xlabel('Motoneurons')
    ax[pooli].set_ylabel('Proportion of common inhibitory inputs')
    ax[pooli].set_title(f'Pool #{pooli}')
    ax[pooli].legend()
plt.tight_layout(rect=[0,0,1,0.96])
plt.suptitle("Distribution of common inhibitory inputs")
new_filename = f'Inhibitory_common_input_distribution_to_MN.png'
save_file_path = os.path.join(new_directory, new_filename)
plt.savefig(save_file_path)
plt.show()

if nb_inhibitory_inputs_per_pool >= 1:
    plt.figure(figsize=(7,15))
    sns.heatmap(inhib_inputs_to_mn_weight_matrix, annot=False, cmap="PiYG", fmt=".2f", linewidths=0.5,
                xticklabels=custom_matrix_labels, vmin=0)
    plt.title("Distribution of common inhibitory inputs - matrix format")
    plt.xlabel("Input #")
    plt.ylabel("Motoneuron #")
    new_filename = f'Inhibitory_common_input_distribution_to_MN_matrix_format.png'
    save_file_path = os.path.join(new_directory, new_filename)
    plt.savefig(save_file_path)
    plt.show()

In [169]:
### MN INDEPENDENT NOISE INPUT (delivered both to the excitatory and inhibitory components)

MN_independent_noise_excit = []
MN_independent_noise_inhib = []
for mni in range(total_nb_motoneurons):
    MN_independent_noise_excit.append(Generate_filtered_gaussian_noise_input(
        low_pass_filter_of_MN_independent_noise, 0, MN_independent_noise_excit_std))
    MN_independent_noise_inhib.append(Generate_filtered_gaussian_noise_input(
        low_pass_filter_of_MN_independent_noise, 0, MN_independent_noise_inhib_std))

### OTHER CELL TYPES INDEPENDENT NOISE INPUT
RC_independent_noise = []
Ib_interneurons_independent_noise = []
total_nb_renshaw_cells = nb_Renshaw_cells_per_pool * nb_pools
for renshawi in range(total_nb_renshaw_cells):
    RC_independent_noise.append(Generate_filtered_gaussian_noise_input(
        low_pass_filter_of_other_cell_types_independent_noise, 0, RC_independent_noise_std))
for Ib_i in range(total_nb_Ib_interneurons):
    Ib_interneurons_independent_noise.append(Generate_filtered_gaussian_noise_input(
        low_pass_filter_of_other_cell_types_independent_noise, 0, Ib_interneruons_noise_std))


In [ ]:
excit_inputs_to_mn_weight_matrix

In [171]:
### Create the Timed Arrays storing the inputs to be used during the simulation #############

# Motor neurons' excitation conductance timed arrays ###
temp_timed_array = np.zeros((int(np.round(duration_with_ignored_window/second*fsamp)), total_nb_motoneurons))
for pooli in range(nb_pools):
    for mni in range(nb_motoneurons_per_pool):
        current_mn = pooli*nb_motoneurons_per_pool + mni
        for inputi in range(nb_excitatory_inputs_per_pool):
            current_input = pooli*nb_excitatory_inputs_per_pool + inputi
            # add each input scaled by the distribution weights
            temp_timed_array[:, current_mn] += np.array(
                MN_excit_input[pooli][inputi]).flatten() * excit_inputs_to_mn_weight_matrix[current_mn, current_input]
        # add independent input
        temp_timed_array[:, current_mn] += MN_independent_noise_excit[current_mn]
        # Add the baseline mean
        temp_timed_array[:, current_mn] += excitatory_input_baseline
        # prevent negative values
        temp_timed_array[:, current_mn] = np.maximum(temp_timed_array[:, current_mn], 0)
input_excit = TimedArray(temp_timed_array * msiemens, dt=(1/fsamp)*second)

# Just a visual check #######
# plt.figure(figsize=(30,6))
# plt.plot(temp_timed_array[:,np.arange(1,10)],alpha=0.2, color = 'C0')
# test = np.mean(temp_timed_array[:,np.arange(1,100)],axis=1)
# plt.plot(test, color='darkblue')


# Motor neurons' inhibitory conductance timed arrays ###
temp_timed_array = np.zeros((int(np.round(duration_with_ignored_window/second*fsamp)), total_nb_motoneurons))
for pooli in range(nb_pools):
    for mni in range(nb_motoneurons_per_pool):
        current_mn = pooli*nb_motoneurons_per_pool + mni
        for inputi in range(nb_inhibitory_inputs_per_pool):
            current_input = pooli*nb_excitatory_inputs_per_pool + inputi
            # add each input scaled by the distribution weights
            temp_timed_array[:, current_mn] += np.array(
                MN_inhib_input[pooli][inputi]).flatten() * inhib_inputs_to_mn_weight_matrix[current_mn, current_input] # add the inhibitory input mean here
        # add independent input
        temp_timed_array[:, current_mn] += MN_independent_noise_inhib[current_mn]
        # Add the baseline mean
        temp_timed_array[:, current_mn] += inhibitory_input_mean
        # prevent negative values
        temp_timed_array[:, current_mn] = np.maximum(temp_timed_array[:, current_mn], 0)
input_inhib = TimedArray(temp_timed_array * msiemens, dt=(1/fsamp)*second)


# Renshaw cell and Ib interneurons independent input timed arrays ###
temp_timed_array = np.zeros((int(np.round(duration_with_ignored_window/second*fsamp)), total_nb_renshaw_cells))
for renshawi in range(total_nb_renshaw_cells):
    temp_timed_array[:, renshawi] += RC_independent_noise[renshawi]
input_RC = TimedArray(temp_timed_array * mvolt, dt=(1/fsamp)*second)

temp_timed_array = np.zeros((int(np.round(duration_with_ignored_window/second*fsamp)), total_nb_Ib_interneurons))
for Ib_i in range(total_nb_Ib_interneurons):
    temp_timed_array[:, Ib_i] += Ib_interneurons_independent_noise[Ib_i]
input_Ib = TimedArray(temp_timed_array * mvolt, dt=(1/fsamp)*second)

In [172]:
# # Just a visual check #######
# plt.figure(figsize=(30,6))
# plt.plot(temp_timed_array[:,np.arange(1,10)],alpha=0.2, color = 'C0')
# test = np.mean(temp_timed_array[:,np.arange(1,100)],axis=1)
# plt.plot(test, color='darkblue')

# plt.figure(figsize=(30,6))
# # plt.plot(input_excit.values[:,1], color='C1')
# plt.plot(input_excit.values * 1e5, color='r', alpha=0.2)
# # plt.axhline(y=np.mean(input_excit.values[:,1]), color='r', linestyle='--')
# plt.plot(np.mean(input_excit.values * 1e5,axis=1), color='r')

# plt.figure(figsize=(30,6))
# # plt.plot(input_inhib.values[:,1])
# plt.plot(input_inhib.values * 1e5, color='b', alpha=0.2)
# # plt.axhline(y=np.mean(input_inhib.values[:,1]), color='r', linestyle='--')
# plt.plot(np.mean(input_inhib.values * 1e5,axis=1), color='b')

In [173]:
# Renshaw cell and Ib interneurons independent input timed arrays ###
temp_timed_array = np.zeros((int(np.round(duration_with_ignored_window/second*fsamp)), total_nb_renshaw_cells))
for renshawi in range(total_nb_renshaw_cells):
    temp_timed_array[:, renshawi] += RC_independent_noise[renshawi]
input_RC = TimedArray(temp_timed_array * mvolt, dt=(1/fsamp)*second)

temp_timed_array = np.zeros((int(np.round(duration_with_ignored_window/second*fsamp)), total_nb_Ib_interneurons))
for Ib_i in range(total_nb_Ib_interneurons):
    temp_timed_array[:, Ib_i] += Ib_interneurons_independent_noise[Ib_i]
input_Ib = TimedArray(temp_timed_array * mvolt, dt=(1/fsamp)*second)

In [ ]:
# CREATE CONNECTIVITY BETWEEN MOTOR NEURONS AND RENSHAW CELLS

# Create connectivity matrix
MN_to_Renshaw_connectivity_matrix = np.zeros([total_nb_motoneurons,total_nb_renshaw_cells])
Renshaw_to_MNs_connectivity_matrix = np.zeros([total_nb_renshaw_cells,total_nb_motoneurons])

# Connections are binary (either zero or ones). Randomly generate the connectivity according to the proportion (probability) of connections

# For each pool of MN, set the list of RCs which are eligible to receive connections from each MN pool
RC_indices_to_consider_relative_to_MN_pool = {}
for pool_rc_i in range(nb_pools):
    RC_indices_to_consider_relative_to_MN_pool[pool_rc_i] = []
    RC_indices_in_current_pool = np.arange(pool_rc_i*nb_Renshaw_cells_per_pool, (pool_rc_i+1)*nb_Renshaw_cells_per_pool)
    if MN_to_Renshaw_connectivity_prevent_heteronymous_pool_overlap:
        # Randomly divide the array of RC indices into N grups (N=nb of pools) of equal size
        RC_indices_in_current_pool_shuffled = np.random.permutation(RC_indices_in_current_pool)
        # Determie split size (number of RCs in each array = useful if the nb of RCs is not divisible by N)
        split_sizes = np.full(nb_pools, len(RC_indices_in_current_pool) // nb_pools)  # Initial equal sizes
        split_sizes[:len(RC_indices_in_current_pool_shuffled) % nb_pools] += 1          # Distribute the remainder among the first splits
        # Split the array
        split_arrays = np.split(RC_indices_in_current_pool, np.cumsum(split_sizes)[:-1])
    for pool_mn_i in range(nb_pools):
        if MN_to_Renshaw_connectivity_prevent_heteronymous_pool_overlap:
            if pool_mn_i != pool_rc_i:
                RC_indices_to_consider_relative_to_MN_pool[pool_rc_i].append(split_arrays[pool_mn_i])
            else:
                RC_indices_to_consider_relative_to_MN_pool[pool_rc_i].append(RC_indices_in_current_pool)
        else:
            RC_indices_to_consider_relative_to_MN_pool[pool_rc_i].append(RC_indices_in_current_pool)
# For each pool of RCs, set the list of MNs which are eligible to receive connections from each RC pool
MN_indices_to_consider_relative_to_RC_pool = {}
for pool_mn_i in range(nb_pools):
    MN_indices_to_consider_relative_to_RC_pool[pool_mn_i] = []
    MN_indices_in_current_pool = np.arange(pool_mn_i*nb_motoneurons_per_pool, (pool_mn_i+1)*nb_motoneurons_per_pool)
    if Renshaw_to_MN_connectivity_prevent_heteronymous_pool_overlap:
        # Randomly divide the array of RC indices into N grups (N=nb of pools) of equal size
        MN_indices_in_current_pool_shuffled = np.random.permutation(MN_indices_in_current_pool)
        # Determie split size (number of RCs in each array = useful if the nb of RCs is not divisible by N)
        split_sizes = np.full(nb_pools, len(MN_indices_in_current_pool) // nb_pools)  # Initial equal sizes
        split_sizes[:len(MN_indices_in_current_pool) % nb_pools] += 1          # Distribute the remainder among the first splits
        # Split the array
        split_arrays = np.split(MN_indices_in_current_pool_shuffled, np.cumsum(split_sizes)[:-1])
    for pool_rc_i in range(nb_pools):
        if Renshaw_to_MN_connectivity_prevent_heteronymous_pool_overlap:
            if pool_mn_i != pool_rc_i:
                MN_indices_to_consider_relative_to_RC_pool[pool_mn_i].append(split_arrays[pool_rc_i])
            else:
                MN_indices_to_consider_relative_to_RC_pool[pool_mn_i].append(MN_indices_in_current_pool)
        else:
            MN_indices_to_consider_relative_to_RC_pool[pool_mn_i].append(MN_indices_in_current_pool)

# Actually set the connectivity
for pool_mn_i in range(nb_pools):
    MN_indices_in_current_pool = np.arange(pool_mn_i*nb_motoneurons_per_pool, (pool_mn_i+1)*nb_motoneurons_per_pool)
    for pool_rc_i in range(nb_pools):
        RC_indices_in_current_pool = np.arange(pool_rc_i*nb_Renshaw_cells_per_pool, (pool_rc_i+1)*nb_Renshaw_cells_per_pool)
        print(f"Creating connections between MNs (cluster {pool_mn_i}) and Renshaw cells (cluster {pool_rc_i})")
        
        if pool_mn_i == pool_rc_i: # within-cluster connectivity (homonymous pools)
            temp_MN_RC_connectivity_probability = MN_to_Renshaw_connectivity_probability_within_pool
            temp_RC_MN_connectivity_probability = Renshaw_to_MN_connectivity_probability_within_pool

        else: # between-cluster connectivity (heteronymous pools)
            temp_MN_RC_connectivity_probability = MN_to_Renshaw_connectivity_probability_across_pool
            temp_RC_MN_connectivity_probability = Renshaw_to_MN_connectivity_probability_across_pool

        # MN to Renshaw connections
        RC_indices_which_can_be_connected_to_MN = RC_indices_to_consider_relative_to_MN_pool[pool_rc_i][pool_mn_i]
        probability_multiplier_temp = (len(RC_indices_which_can_be_connected_to_MN)/nb_Renshaw_cells_per_pool)**(-1)
        nb_of_MN_to_Renshaw_connection = int(probabilistic_round(len(RC_indices_which_can_be_connected_to_MN)*temp_MN_RC_connectivity_probability*probability_multiplier_temp ))
        nb_of_MN_to_Renshaw_connection = np.min([nb_of_MN_to_Renshaw_connection, len(RC_indices_which_can_be_connected_to_MN)])
        for mni in MN_indices_in_current_pool:
            temp_idx_of_MN_to_Renshaw_connection = np.random.choice(RC_indices_which_can_be_connected_to_MN, nb_of_MN_to_Renshaw_connection, replace=False)
            MN_to_Renshaw_connectivity_matrix[mni,temp_idx_of_MN_to_Renshaw_connection] = 1

        # Renshaw to MN connections
        MN_indices_which_can_be_connected_to_Renshaw = MN_indices_to_consider_relative_to_RC_pool[pool_mn_i][pool_rc_i]
        probability_multiplier_temp = (len(MN_indices_which_can_be_connected_to_Renshaw)/nb_motoneurons_per_pool)**(-1)
        nb_of_Renshaw_to_MN_connection = int(probabilistic_round(len(MN_indices_which_can_be_connected_to_Renshaw)*temp_RC_MN_connectivity_probability*probability_multiplier_temp ))
        nb_of_Renshaw_to_MN_connection = np.min([nb_of_Renshaw_to_MN_connection, len(MN_indices_which_can_be_connected_to_Renshaw)])
        for rci in RC_indices_in_current_pool:
            temp_idx_of_Renshaw_to_MN_connection = np.random.choice(MN_indices_which_can_be_connected_to_Renshaw, nb_of_Renshaw_to_MN_connection, replace=False)
            Renshaw_to_MNs_connectivity_matrix[rci,temp_idx_of_Renshaw_to_MN_connection] = 1

# Function to truncate a colormap
def truncate_colormap(cmap, min_val=0.0, max_val=0.5, n=100):
    """Truncate a colormap to use only a portion of it"""
    new_cmap = mcolors.LinearSegmentedColormap.from_list(
        f'trunc({cmap.name},{min_val},{max_val})',
        cmap(np.linspace(min_val, max_val, n))
    )
    return new_cmap

original_cmap = plt.cm.YlOrRd_r
truncated_cmap = truncate_colormap(original_cmap, 0.4, 0.7)

# Plot the connectivity arrays
plt.figure(figsize=(10,10))
plt.imshow(MN_to_Renshaw_connectivity_matrix, vmin = 0.0, vmax = 1.0, cmap=truncated_cmap)
# plt.colorbar()
plt.title(f"MN (y axis) to Renshaw (x axis) connectivity \n dark = unconnected, light = connected \n Mean MN to RC connectivity = {np.mean(MN_to_Renshaw_connectivity_matrix):.2f}")
plt.xlabel("Renshaw cells")
plt.ylabel("Motoneurons")
plt.xticks([])
plt.yticks([])
# plt.axis('off')
new_filename = f'MN_to_RC_connectivity.png'
save_file_path = os.path.join(new_directory, new_filename)
plt.savefig(save_file_path)
plt.show()

original_cmap = plt.cm.BuPu_r
truncated_cmap = truncate_colormap(original_cmap, 0.1, 0.6)

plt.figure(figsize=(10,10))
plt.imshow(Renshaw_to_MNs_connectivity_matrix, vmin = 0.0, vmax = 1.0, cmap=truncated_cmap)
# plt.colorbar()
plt.title(f"Renshaw (x axis) to MN (y axis) connectivity \n dark = unconnected, light = connected \n Mean Renshaw to MN connectivity = {np.mean(Renshaw_to_MNs_connectivity_matrix):.2f}")
plt.ylabel("Renshaw cells")
plt.xlabel("Motoneurons")
plt.xticks([])
plt.yticks([])
# plt.axis('off')
new_filename = f'RC_to_MN_connectivity.png'
save_file_path = os.path.join(new_directory, new_filename)
plt.savefig(save_file_path)
plt.show()

In [ ]:
# CREATE CONNECTIVITY BETWEEN Ib INHIBITORY INTERNEURONS AND MOTOR NEURONS

# Create connectivity matrix
Ib_to_MNs_connectivity_matrix = np.zeros([total_nb_Ib_interneurons,total_nb_motoneurons])

if not Ib_inh_int_to_MN_connectivity_gaussian_nonbinary_weights:
    # Connections are binary (either zero or ones). Randomly generate the connectivity according to the proportion (probability) of connections
    nb_of_Ib_to_MN_connection = int(probabilistic_round(total_nb_motoneurons*Ib_inh_int_to_MN_connectivity_probability))
    for Ib_i in range(total_nb_Ib_interneurons):
        # Ib inhibitory interneurons to MN connections
        temp_idx_of_Ib_to_MN_connection = np.random.choice(np.arange(total_nb_motoneurons), nb_of_Ib_to_MN_connection, replace=False)
        Ib_to_MNs_connectivity_matrix[Ib_i,temp_idx_of_Ib_to_MN_connection] = 1
else:
    for Ib_i in range(total_nb_Ib_interneurons):
        # Ib inhibitory interneurons to MN connections
        temp_weight_of_Ib_to_MN_connection = np.random.normal(Ib_inh_int_to_MN_connectivity_probability, Ib_inh_int_to_MN_connectivity_probability_std, total_nb_motoneurons)
        temp_weight_of_Ib_to_MN_connection = np.clip(temp_weight_of_Ib_to_MN_connection, 0.0, 1.0)
        Ib_to_MNs_connectivity_matrix[Ib_i,:] = temp_weight_of_Ib_to_MN_connection

original_cmap = plt.cm.viridis
truncated_cmap = truncate_colormap(original_cmap, 0.2, 0.8)

# Plot the connectivity arrays
plt.figure(figsize=(10,10))
plt.imshow(Ib_to_MNs_connectivity_matrix, vmin = 0.0, vmax = 1.0, cmap=truncated_cmap)
# plt.colorbar()
plt.title(f"Ib interneurons (y axis) to MN (x axis) connectivity \n dark = unconnected, light = connected \n Mean Ib to MN connectivity = {np.mean(Ib_to_MNs_connectivity_matrix):.2f}")
plt.xlabel("Motoneurons")
plt.ylabel("Ib inhibitory interneurons")
plt.xticks([])
plt.yticks([])
# plt.axis('off')
new_filename = f'Ib_to_MN_connectivity.png'
save_file_path = os.path.join(new_directory, new_filename)
plt.savefig(save_file_path)
plt.show()

In [ ]:
# # # # # Create neuron groups and synapses # # # # #

# # # NEURON GROUPS # # # 

# MOTOR NEURONS #
motoneurons = NeuronGroup(
    total_nb_motoneurons, 
    MN_equations, 
    threshold='v > voltage_thresh', 
    reset='v = voltage_rest',
    refractory='refractory_period',
    method='euler'
)
motoneurons.v = voltage_rest # in mV #
motoneurons.tau = motoneurons_taus * ms # in milliseconds
motoneurons.g_leak = motoneurons_membrane_conductance * msiemens # in milisiemens
motoneurons.C_m = motoneuron_capacitances * ufarad # in microfarads
motoneurons.I_th = motoneurons_rheobases * rheobase_scaling * nA # in nanoAmperes
motoneurons.refractory_period = motoneurons_refractory_periods * ms  # in milliseconds
motoneurons.input_weight = motoneuron_input_weights # dimensionless unit

# MUSCLE FIBERS #
# Define the conductance-based muscle fiber model
muscle_fibers = NeuronGroup(
    total_nb_motoneurons, 
    muscle_fibers_equations, 
    method='euler'
)
muscle_fibers.force = fiber_force_rest  # Initialize force
muscle_fibers.resting_force = fiber_force_rest
muscle_fibers.g_force_rise = 0 * siemens  # Initialize " force conductance"
muscle_fibers.g_force_decay = (muscle_fibers_twitch_rise_taus / muscle_fibers_twitch_decay_taus)*force_baseline_conductance_scalar # Initialize "leak conductance" (force decay)
muscle_fibers.force_twitch_rise_tau = muscle_fibers_twitch_rise_taus* second
# muscle_fibers.force_decay_tau = muscle_fibers_twitch_decay_taus * second  # Initialize force decay time constant
muscle_fibers.fiber_max_force = muscle_fibers_max_force * volt


# TOTAL FORCE #
# total force "neuron"
total_force_neuron = NeuronGroup(1, total_muscle_force_equations, method='euler')
total_force_neuron.force_total = 0 * volt  # Initialize total force

# RENSHAW CELLS #
renshaw_cells = NeuronGroup(
    total_nb_renshaw_cells,
    RC_equations,
    threshold='v > voltage_thresh', 
    reset='v = voltage_rest',
    refractory='refractory_period_RC',
    method='euler'
)
renshaw_cells.v = voltage_rest  # Initialize membrane potential
renshaw_cells.tau = tau_Renshaw

# Ib INHIBITORY INTERNEURONS #
Ib_inhibitory_interneurons = NeuronGroup(
    total_nb_Ib_interneurons,
    Ib_interneurons_equations,
    threshold='v > voltage_thresh', 
    reset='v = voltage_rest',
    refractory='refractory_period_Ib_interneurons',
    method='euler'
)
Ib_inhibitory_interneurons.v = voltage_rest  # Initialize membrane potential
Ib_inhibitory_interneurons.tau = tau_Ib_interneurons


# # # SYNAPSES # # # 

# MOTOR NEURONS => RENSHAW CELLS SYNAPSES and VICE-VERSA #
# Connect motor neurons to Renshaw cells
synapses_MN_to_Renshaw = Synapses(motoneurons, renshaw_cells, 'w : 1',
                           on_pre='v += MN_to_Renshaw_excit*w',
                           delay = MN_RC_synpatic_delay)
pre_indices, post_indices = np.nonzero(MN_to_Renshaw_connectivity_matrix)
weights_to_assign = MN_to_Renshaw_connectivity_matrix[pre_indices,post_indices]
synapses_MN_to_Renshaw.connect(i=pre_indices, j=post_indices)
synapses_MN_to_Renshaw.w = weights_to_assign

synapses_Renshaw_to_MN = Synapses(renshaw_cells, motoneurons, 'w : 1',
                            on_pre='gi += Renshaw_to_MN_inhib * w * input_weight',
                           delay = MN_RC_synpatic_delay)
pre_indices, post_indices = np.nonzero(Renshaw_to_MNs_connectivity_matrix)
weights_to_assign = Renshaw_to_MNs_connectivity_matrix[pre_indices,post_indices]
synapses_Renshaw_to_MN.connect(i=pre_indices, j=post_indices)
synapses_Renshaw_to_MN.w = weights_to_assign


# MOTOR NEURONS => MUSCLE FIBERS SYNAPSES #
# Connect motor neurons to muscle fibers
synapses_MN_to_muscle_fibers = Synapses(motoneurons, muscle_fibers, 
    on_pre='g_force_rise += MN_to_muscle_fiber_force * force_baseline_conductance_scalar',  # Increase conductance on a spike
    method='euler'
)
synapses_MN_to_muscle_fibers.connect(j='i')  # One-to-one connection
# Assign delays from the numpy array
synapses_MN_to_muscle_fibers.delay = muscle_fibers_electromechanical_delay * second

# MUSCLE FIBERS => TOTAL MUSCLE FORCE SYNAPSES #
# Connect muscle fibers to total muscle force
synapses_muscle_fibers_to_total_force = Synapses(muscle_fibers, total_force_neuron,
    '''force_total_post = force_pre : volt (summed)''',
    method='euler'
)
synapses_muscle_fibers_to_total_force.connect(i=np.arange(total_nb_motoneurons), j=0)

# TOTAL MUSCLE FORCE SYNAPSES => Ib INHIBITORY INTERNEURONS SYNAPSES #
# Connect muscle fibers to total muscle force
synapses_total_force_to_Ib_interneurons = Synapses(
    total_force_neuron, Ib_inhibitory_interneurons,
    '''force_induced_current_post = force_total_relative_pre * force_to_Ib_excit : volt (summed)''',  #1/R to get siemens and get units to match
    method='euler'
)
synapses_total_force_to_Ib_interneurons.connect(i=0, j=np.arange(total_nb_Ib_interneurons))

# Ib INHIBITORY INTERNEURONS => MOTOR NEURONS SYNAPSES #
# Connect motor neurons to Renshaw cells
synapses_Ib_to_MN = Synapses(Ib_inhibitory_interneurons, motoneurons, 'w : 1',
                           on_pre='gi += Ib_to_MN_inhib * w * input_weight',
                           delay = Ib_to_MN_synaptic_delay)
pre_indices, post_indices = np.nonzero(Ib_to_MNs_connectivity_matrix)
weights_to_assign = Ib_to_MNs_connectivity_matrix[pre_indices,post_indices]
synapses_Ib_to_MN.connect(i=pre_indices, j=post_indices)
synapses_Ib_to_MN.w = weights_to_assign

In [177]:
#### RUN SIMULATION

# Initialize values
motoneurons.v = voltage_rest # in mV
muscle_fibers.force = fiber_force_rest  # Initialize force
muscle_fibers.g_force_rise = 0 * siemens  # Initialize " force conductance"
total_force_neuron.force_total = 0 * volt  # Initialize total force
renshaw_cells.v = voltage_rest  # Initialize membrane potential
Ib_inhibitory_interneurons.v = voltage_rest  # Initialize membrane potential

# Set monitors
monitor_spikes_motoneurons = SpikeMonitor(motoneurons, record=True)
monitor_voltage_motoneurons = StateMonitor(motoneurons, 'v', record=True) # get the voltage trace
monitor_spikes_renshaw_cells = SpikeMonitor(renshaw_cells, record=True)
fiber_force_monitor = StateMonitor(muscle_fibers, 'force', record=True)
total_force_monitor = StateMonitor(total_force_neuron, 'force_total', record=True)
monitor_Ib_interneurons = SpikeMonitor(Ib_inhibitory_interneurons, record=True)

# Run the simulation
run(duration_with_ignored_window)

In [178]:
# GET SPIKE TRAINS AND BINARY SPIKE TRAINS
def Get_binary_spike_trains(spike_monitor, sim_duration):
    # Define time bins
    time_bins = np.arange(0, int(np.round((sim_duration*fsamp)))*second) * ((1/fsamp)*second)

    # Retrieve spikes and get binary spike trains
    spike_trains = []
    for mni in range(total_nb_motoneurons):
        spike_trains.append(spike_monitor.spike_trains()[mni])
    
    # Initialize the binary spike train array
    binary_spike_trains = {}
    binary_spike_trains = np.zeros((total_nb_motoneurons, len(time_bins)))
    # Convert spike times to binary spike train
    for neuron_idx in range(total_nb_motoneurons):
        spikes = spike_trains[neuron_idx]
        spike_indices = np.searchsorted(time_bins, spikes)
        binary_spike_trains[neuron_idx, spike_indices-1] = 1 #-1 because of offset due to 0-indexing
    
    return spike_trains, binary_spike_trains

motoneurons_spike_trains, motoneurons_binary_spike_trains = Get_binary_spike_trains(monitor_spikes_motoneurons, duration_with_ignored_window)

In [179]:
# # Plot results - voltage trace of first MN
# plt.figure(figsize=(40, 6))
# plt.plot(monitor_voltage_motoneurons.t / ms, monitor_voltage_motoneurons.v[0]/volt*1000, color='blue', label='MN #0 voltage')
# plt.axhline(voltage_thresh/volt*1000, color='red', linestyle='--', label='Threshold voltage')
# plt.xlabel('Time (ms)')
# plt.ylabel('Voltage (mV)')
# plt.title('Voltage trace')
# plt.legend()
# plt.show()

In [ ]:
### PLOT SPIKE TRAINS RESULTS - RENSHAW CELLS AND IB INTERNEURONS

plt.figure(figsize=(20, 10))
plt.plot(monitor_spikes_renshaw_cells.t / ms, monitor_spikes_renshaw_cells.i, '.b', alpha = 0.1, markersize=10)
plt.xlabel('Time (ms)')
plt.ylabel('Renshaw cell index')
plt.title('RC spike raster plot')
new_filename = f'RC_firings_raster_plot.png'
save_file_path = os.path.join(new_directory, new_filename)
plt.savefig(save_file_path)
plt.show()

plt.figure(figsize=(20, 10))
plt.plot(monitor_Ib_interneurons.t / ms, monitor_Ib_interneurons.i, '.g', alpha = 0.1, markersize=10)
plt.xlabel('Time (ms)')
plt.ylabel('Ib interneuron index')
plt.title('Ib interneuron spike raster plot')
new_filename = f'Ib_interneurons_firings_raster_plot.png'
save_file_path = os.path.join(new_directory, new_filename)
plt.savefig(save_file_path)
plt.show()

In [ ]:
### Get discharge characteristics of Renshaw cells
spike_trains_RC = {}
firing_rates_RC = {}
mean_firing_rate_RC = {}
std_firing_rate_RC = {}
fig, axs = plt.subplots(1,nb_pools)
if nb_pools==1:
    axs=[axs]
for pooli in range(nb_pools):
    # Retrieve spikes
    spike_trains_RC[pooli] = []
    for rci in range(nb_Renshaw_cells_per_pool):
        spike_trains_RC[pooli].append(monitor_spikes_renshaw_cells.spike_trains()[pooli*nb_Renshaw_cells_per_pool+rci])
    # Calculate the firing rate for each Renshaw cell
    firing_rates_RC[pooli] = []
    for rci in range(nb_Renshaw_cells_per_pool):
        firing_rate_temp = len(spike_trains_RC[pooli][rci]) / duration_with_ignored_window
        firing_rates_RC[pooli].append(firing_rate_temp)
    # Convert to a numpy array for easier calculations
    firing_rates_RC[pooli] = np.array(firing_rates_RC[pooli])
    # Calculate mean and standard deviation of the firing rates
    mean_firing_rate_RC[pooli] = np.mean(firing_rates_RC[pooli])
    std_firing_rate_RC[pooli] = np.std(firing_rates_RC[pooli])
    # Renshaw cells' firing rates results
    print(f"Mean firing rate of Renshaw cells (pool #{pooli}): {mean_firing_rate_RC[pooli]:.2f} Hz")
    print(f"Standard deviation of firing rates of Renshaw cells (pool #{pooli}): {std_firing_rate_RC[pooli]:.2f} Hz")
    axs[pooli].hist(firing_rates_RC[pooli], edgecolor='white', color=pool_colors_one[pooli], alpha=0.75)
    axs[pooli].axvline(x = mean_firing_rate_RC[pooli], color = pool_colors_one[pooli], linestyle='--', linewidth=2, label='Mean firing rate')
    axs[pooli].set_xlabel("Mean firing rate (pps)")
    axs[pooli].set_ylabel("Renshaw cell count")
    axs[pooli].set_title(f"Pool #{pooli}")
plt.tight_layout(rect=[0,0,1,0.96])
plt.suptitle("Histogram of renshaw cells' firing rate")
new_filename = f'Hist_RC_Discharge_rates.png'
save_file_path = os.path.join(new_directory, new_filename)
plt.savefig(save_file_path)
plt.show(fig)



In [ ]:
### Get discharge characteristics of Ib interneurons
spike_trains_Ib = []
firing_rates_Ib = []
mean_firing_rate_Ib = []
std_firing_rate_Ib = []

plt.figure()
# Retrieve spikes
for Ib_i in range(total_nb_Ib_interneurons):
    spike_trains_Ib.append(monitor_Ib_interneurons.spike_trains()[Ib_i])
    firing_rate_temp = len(spike_trains_Ib[Ib_i]) / duration_with_ignored_window
    firing_rates_Ib.append(firing_rate_temp)
# Convert to a numpy array for easier calculations
firing_rates_Ib = np.array(firing_rates_Ib)
# Calculate mean and standard deviation of the firing rates
mean_firing_rate_Ib = np.mean(firing_rates_Ib)
std_firing_rate_Ib = np.std(firing_rates_Ib)
# Renshaw cells' firing rates results
print(f"Mean firing rate of Ib interneurons: {mean_firing_rate_Ib:.2f} Hz")
print(f"Standard deviation of firing rates of Ib interneurons: {std_firing_rate_Ib:.2f} Hz")
plt.hist(firing_rates_Ib, edgecolor='white', color="green", alpha=0.75)
plt.axvline(x = mean_firing_rate_Ib, color = "green", linestyle='--', linewidth=2, label='Mean firing rate')
plt.xlabel("Mean firing rate (pps)")
plt.ylabel("Ib interneurons count")
plt.title(f"Ib interneurons firing rate")

new_filename = f'Hist_Ib_Discharge_rates.png'
save_file_path = os.path.join(new_directory, new_filename)
plt.savefig(save_file_path)
plt.show(fig)


In [ ]:
max_ISI_per_MN = np.zeros(total_nb_motoneurons)
mean_ISI_per_MN = np.zeros(total_nb_motoneurons)
std_ISI_per_MN = np.zeros(total_nb_motoneurons)
max_DR_per_MN = np.zeros(total_nb_motoneurons)
mean_DR_per_MN = np.zeros(total_nb_motoneurons)
std_DR_per_MN = np.zeros(total_nb_motoneurons)
valid_MU_idx = np.zeros(total_nb_motoneurons, dtype=bool)
# Create a figure with 2 subplots = one for the ISI, one for the frequency
fig, axs = plt.subplots(2, 2, figsize=(15, 10))  # 2 row, 2 columns
for mni in range(total_nb_motoneurons):
    # Modifying the spike train to consider only the time window of interest
    idx_to_keep1 = motoneurons_spike_trains[mni] > (window_beginning_ignore*second)
    idx_to_keep2 = motoneurons_spike_trains[mni] < (duration_with_ignored_window-(window_end_ignore*second))
    idx_to_keep = idx_to_keep1 * idx_to_keep2
    if idx_to_keep.sum() > 1:
        temp_spike_train = motoneurons_spike_trains[mni][idx_to_keep]
        temp_ISI = np.diff(temp_spike_train/second)
        max_ISI_per_MN[mni] = np.max(temp_ISI)
        mean_ISI_per_MN[mni] = np.mean(temp_ISI)
        std_ISI_per_MN[mni] = np.std(temp_ISI)
        max_DR_per_MN[mni] = np.max(1/temp_ISI)
        mean_DR_per_MN[mni] = np.mean(1/temp_ISI)
        std_DR_per_MN[mni] = np.std(1/temp_ISI)
    else:
        max_ISI_per_MN[mni] = np.nan
        mean_ISI_per_MN[mni] = np.nan
        std_ISI_per_MN[mni] = np.nan
        max_DR_per_MN[mni] = np.nan
        mean_DR_per_MN[mni] = np.nan
        std_DR_per_MN[mni] = np.nan
    if max_ISI_per_MN[mni] < ISI_threshold_for_discontinuity:
        valid_MU_idx[mni] = True
# valid_MU_idx = np.where(valid_MU_idx)[0]

# Subplot 1: ISI ~ Motoneuron Index
axs[0,0].plot(np.arange(total_nb_motoneurons),max_ISI_per_MN, color = 'black', linestyle = '--', label="Max ISI")
axs[0,0].plot(np.arange(total_nb_motoneurons),mean_ISI_per_MN, color = 'C0', label="Mean ISI")
axs[0, 0].fill_between(
    np.arange(total_nb_motoneurons),
    mean_ISI_per_MN - std_ISI_per_MN,
    mean_ISI_per_MN + std_ISI_per_MN,
    color='C0',
    alpha=0.2,
    label="Mean ISI ± Std"
)
axs[0,0].axhline(y=ISI_threshold_for_discontinuity, color = 'blue', alpha = 0.5, label="ISI threshold for discontinuity")
axs[0,0].set_xlim(-1,total_nb_motoneurons)
axs[0,0].set_ylabel("Inter-spike interval (s)")
axs[0,0].set_xlabel("Motoneuron index")
axs[0,0].set_title(f"Inter-spike interval ~ motor neuron index")

# Subplot 2: DR ~ Motoneuron Index
axs[1,0].plot(np.arange(total_nb_motoneurons),max_DR_per_MN, color = 'black', linestyle = '--',  label="Max DR")
axs[1,0].plot(np.arange(total_nb_motoneurons),mean_DR_per_MN, color = 'C1',  label="Mean DR")
axs[1, 0].fill_between(
    np.arange(total_nb_motoneurons),
    mean_DR_per_MN - std_DR_per_MN,
    mean_DR_per_MN + std_DR_per_MN,
    color='C1',
    alpha=0.2,
    label="Mean DR ± Std"
)
# Mask invalid values
mean_DR_masked = np.ma.masked_where(~valid_MU_idx, mean_DR_per_MN)
std_DR_masked = np.ma.masked_where(~valid_MU_idx, std_DR_per_MN)
axs[1,0].plot(np.arange(total_nb_motoneurons),mean_DR_masked, color = 'red', linewidth=2,  label="Mean DR (only valid MUs)")
axs[1, 0].fill_between(
    np.arange(total_nb_motoneurons),
    mean_DR_masked - std_DR_masked,
    mean_DR_masked + std_DR_masked,
    color='red',
    alpha=0.2,
    label="Mean DR ± Std (only valid MUs)"
)
axs[1,0].axhline(y=np.mean(mean_DR_per_MN[valid_MU_idx]), color = 'red', alpha = 0.5,
                 label=f"Mean DR of valid MUs = {np.mean(mean_DR_per_MN[valid_MU_idx]):.1f}+-{np.std(mean_DR_per_MN[valid_MU_idx]):.1f} pps")
axs[1,0].set_xlim(-1,total_nb_motoneurons)
axs[1,0].set_ylabel("Firing rate (pps)")
axs[1,0].set_xlabel("Motoneuron index")
axs[1,0].set_title(f"Firing rate ~ motor neuron index")

# Subplot 3: ISI ~ Motoneuron Size
first_pool_indices = np.arange(nb_motoneurons_per_pool)
axs[0,1].plot(motoneuron_normalized_soma_diameters[first_pool_indices],max_ISI_per_MN[first_pool_indices], color = 'black', linestyle = '--', label="Max ISI")
axs[0,1].plot(motoneuron_normalized_soma_diameters[first_pool_indices],mean_ISI_per_MN[first_pool_indices], color = 'C0', label="Mean ISI")
axs[0, 1].fill_between(
    motoneuron_normalized_soma_diameters[first_pool_indices],
    mean_ISI_per_MN[first_pool_indices] - std_ISI_per_MN[first_pool_indices],
    mean_ISI_per_MN[first_pool_indices] + std_ISI_per_MN[first_pool_indices],
    color='C0',
    alpha=0.2,
    label="Mean ISI ± Std"
)
axs[0,1].axhline(y=ISI_threshold_for_discontinuity, color = 'blue', alpha = 0.5, label="ISI threshold for discontinuity")
axs[0,1].set_xlim(-0.02,1.02)
axs[0,1].set_ylabel("Inter-spike interval (s)")
axs[0,1].set_xlabel("Motoneuron size (0 = smallest, 1 = largest)")
axs[0,1].set_title(f"Inter-spike interval ~ motor neuron relative size (only first pool)")

# Subplot 4: DR ~ Motoneuron Size
axs[1,1].plot(motoneuron_normalized_soma_diameters[first_pool_indices],max_DR_per_MN[first_pool_indices], color = 'black', linestyle = '--', label="Max DR")
axs[1,1].plot(motoneuron_normalized_soma_diameters[first_pool_indices],mean_DR_per_MN[first_pool_indices], color = 'C1', label="Mean DR")
axs[1, 1].fill_between(
    motoneuron_normalized_soma_diameters[first_pool_indices],
    mean_DR_per_MN[first_pool_indices] - std_DR_per_MN[first_pool_indices],
    mean_DR_per_MN[first_pool_indices] + std_DR_per_MN[first_pool_indices],
    color='C1',
    alpha=0.2,
    label="Mean DR ± Std"
)
mean_DR_masked = np.ma.masked_where(~valid_MU_idx[first_pool_indices], mean_DR_per_MN[first_pool_indices])
std_DR_masked = np.ma.masked_where(~valid_MU_idx[first_pool_indices], std_DR_per_MN[first_pool_indices])
axs[1,1].plot(motoneuron_normalized_soma_diameters[first_pool_indices],mean_DR_masked, color = 'red', linewidth=2,  label="Mean DR (only valid MUs)")
axs[1, 1].fill_between(
    motoneuron_normalized_soma_diameters[first_pool_indices],
    mean_DR_masked - std_DR_masked,
    mean_DR_masked + std_DR_masked,
    color='red',
    alpha=0.2,
    label="Mean DR ± Std (only valid MUs)"
)
axs[1,1].axvline(x=np.max(motoneuron_normalized_soma_diameters[first_pool_indices][valid_MU_idx[first_pool_indices]]),
                 color = 'red', alpha = 0.5, label=f"Relative size of the largest valid motor neuron = {np.max(motoneuron_normalized_soma_diameters[first_pool_indices][valid_MU_idx[first_pool_indices]])*100:.0f}%")
axs[1,1].set_xlim(-0.02,1.02)
axs[1,1].set_ylabel("Firing rate (pps)")
axs[1,1].set_xlabel("Motoneuron size (0 = smallest, 1 = largest)")
axs[1,1].set_title(f"Firing rate ~ motor neuron relative size (only first pool)")

for i, ax in enumerate(axs.flat):
    ax.legend()
plt.tight_layout()
new_filename = f'MN_firing_rates_and_ISI.png'
save_file_path = os.path.join(new_directory, new_filename)
plt.savefig(save_file_path)
plt.show()

In [ ]:
### PLOT SPIKE TRAINS RESULTS - MOTOR NEURONS

# Raster plot of motor neuron discharge times
plt.figure(num=1,figsize=(20,10))
mn_iter_i = -1
for pooli in range(nb_pools):
    colormap_temp = cm.get_cmap(pool_cmap[pooli])
    for mni in range(nb_motoneurons_per_pool):
        mn_iter_i += 1
        if mn_iter_i in np.where(valid_MU_idx)[0]:
            temp_alpha = 0.5
        else:
            temp_alpha = 0.2
        plt.scatter((motoneurons_spike_trains[mn_iter_i]/second)*fsamp, np.ones(len(motoneurons_spike_trains[mn_iter_i]))*mn_iter_i,
                    color=colormap_temp(0.3+(mni/(nb_motoneurons_per_pool-1))/1.5), linewidth = 0.2, alpha = temp_alpha)
plt.axvline(samples_to_consider[0],color='black',label='Start of the analyzis window')
plt.axvline(samples_to_consider[-1],color='black',label='End of the analyzis window')
plt.xlabel('Time (s)')
plt.ylabel('Motoneuron index')
plt.title("Raster plot of motoneuron spikes \n Opaque = continuous MU; transparent = discontinuous MU")
plt.legend()
new_filename = f'MN_firings_raster_plot.png'
save_file_path = os.path.join(new_directory, new_filename)
plt.savefig(save_file_path)
plt.show()

In [185]:
# Downsample the data to the same interval as fsamp
downsample_factor = int((1/fsamp) / 1e-4)  # Ratio of original dt to desired dt

if not skip_per_MU_force_computation:
    fiber_force_data = np.zeros((total_nb_motoneurons, int(duration_with_ignored_window*fsamp/second)))
    for fiberi in range(total_nb_motoneurons):
        temp_fiber_force_data = fiber_force_monitor.force[fiberi]/volt
        temp_fiber_force_data = temp_fiber_force_data[::downsample_factor]  # Downsampled 2D array
        fiber_force_data[fiberi,:] = temp_fiber_force_data

# Convert the monitored data to a numpy array for processing
force_total_data = np.squeeze(total_force_monitor.force_total)
force_total_data = np.array(force_total_data[::downsample_factor])


In [ ]:

plt.figure(figsize=(20, 15))
temp_total_force = (force_total_data / muscle_maximal_force) * 100 # in % of MVC
mean_force_during_contraction = np.mean(temp_total_force[samples_to_consider])
if not skip_per_MU_force_computation:
    total_force_to_plot = np.zeros(len(temp_total_force))
    fiberi = -1
    for pooli in range(nb_pools):
        colormap_temp = cm.get_cmap(pool_cmap[pooli])
        for mni in range(nb_motoneurons_per_pool):
            fiberi += 1
            total_force_to_plot += (fiber_force_data[fiberi] / muscle_maximal_force) * 100
            plt.plot(total_force_to_plot, color=colormap_temp(mni/nb_motoneurons_per_pool), lw=1)
plt.plot(temp_total_force, color='black', label = 'total force', lw=2)
# plt.axhline(y=100, color='black', linestyle='--', label='muscle absolute maximal force (100% MVC)')
plt.axhline(y=20, color='blue', linestyle='--', label='Target force (20% MVC)')
plt.axhline(y=mean_force_during_contraction, color='red', alpha = 0.5, linestyle='--', label=f'contraction mean force = {mean_force_during_contraction:.1f}% MVC')
plt.axvline(x=samples_to_consider[0], color='black', linewidth=3, alpha = 0.3, label='start of the window')
plt.axvline(x=samples_to_consider[-1], color='black', linewidth=3, alpha = 0.3, label='end of the window')
plt.xlabel('Time (ms)')
plt.ylabel('Force (% MVC)')
plt.title('Force trace')
plt.ylim(0,25)
plt.legend()
new_filename = f'Total_muscle_force.png'
save_file_path = os.path.join(new_directory, new_filename)
plt.savefig(save_file_path)
plt.show()

normalized_force = force_total_data[samples_to_consider]
normalized_force = (normalized_force / np.mean(normalized_force)) - 1
print(f'Normalized force standard deviation = {np.std(normalized_force)*100:.0f}% of the mean force')

plt.figure(figsize=(10, 5))
plt.plot(normalized_force, color='red', label='Normalized force')
plt.xlabel("Time (samples)")
plt.ylabel("Normalized force")
plt.title(f"Normalized force \n Normalized force standard deviation = {np.std(normalized_force)*100:.2f}% of the mean force")
new_filename = f'Total_muscle_force_normalized_and_variability.png'
save_file_path = os.path.join(new_directory, new_filename)
plt.savefig(save_file_path)
plt.show()

In [ ]:
## SMOOTHING SPIKE TRAINS

Wind_s = 0.4  # hanning window duration. 0.4 for 2.5hz low-pass, 0.2 for 5hz low-pass
HanningW = 2 / round(fsamp * Wind_s) * windows.hann(round(fsamp * Wind_s))  # unitary area

# Filter all valid motor units
smoothed_MN_firing_rates = []
# smoothed_MN_firing_rates_only_valid_MUs_and_samples = []
for mni in range(total_nb_motoneurons):
    smoothed_MN_firing_rates.append(filtfilt(HanningW, 1, motoneurons_binary_spike_trains[mni, :] * fsamp))
    # if mni in np.where(valid_MU_idx)[0]:
    #     smoothed_MN_firing_rates_only_valid_MUs_and_samples.append(smoothed_MN_firing_rates[-1][samples_to_consider])
smoothed_MN_firing_rates = np.array(smoothed_MN_firing_rates)
# smoothed_MN_firing_rates_only_valid_MUs_and_samples = np.array(smoothed_MN_firing_rates_only_valid_MUs_and_samples)

fig, axs = plt.subplots(nb_pools, 1, figsize=(20, 3+(4*nb_pools)))
if nb_pools == 1:
    axs = [axs]
mn_iter_i = -1
for pooli in range(nb_pools):
    colormap_temp = cm.get_cmap(pool_cmap[pooli]) # Getting a smooth color blend from a given colormap
    for mni in range(nb_motoneurons_per_pool):
        mn_iter_i += 1
        if mn_iter_i in np.where(valid_MU_idx)[0]:
            axs[nb_pools-pooli-1].plot(samples_to_consider, smoothed_MN_firing_rates[mn_iter_i, :][samples_to_consider], color=colormap_temp(mni / nb_motoneurons_per_pool), alpha=0.5)
    axs[nb_pools-pooli-1].set_ylabel("Smoothed discharge rate (pps)")
    axs[nb_pools-pooli-1].set_xlabel("Time (samples)")
    # axs[pooli].axvline(samples_to_consider[0],color='black',label='Start of the analyzis window')
    # axs[pooli].axvline(samples_to_consider[-1],color='black',label='End of the analyzis window')
    axs[nb_pools-pooli-1].set_title(f"Pool #{pooli+1}")
    axs[nb_pools-pooli-1].set_ylim(0)
plt.suptitle(f"Smoothed signals of only continuous MUs \n (dark = small MNs ; light = large MNs)")
plt.tight_layout()
new_filename = f'Smoothed_discharge_rates_only_continuous_MUs.png'
save_file_path = os.path.join(new_directory, new_filename)
plt.savefig(save_file_path)
plt.show(fig)

    

In [188]:
## DIMENSIONALITY REDUCTION

# Prepare the data

max_nb_of_factors_to_extract = 10
loadings = []
scores = []
reconstructed_Rsquared = []

if factor_analysis_on_all_pools:
    # Consider MUs from all pools
    idx_of_MUs_to_consider = np.where(valid_MU_idx)[0]
else:
    # Restrict analysis to the first pool
    idx_of_MUs_to_consider = valid_MU_idx[np.arange(nb_motoneurons_per_pool)] 
    idx_of_MUs_to_consider = np.where(idx_of_MUs_to_consider)[0]

smoothed_MN_firing_rates_for_FA = smoothed_MN_firing_rates[idx_of_MUs_to_consider,:]
smoothed_MN_firing_rates_for_FA = smoothed_MN_firing_rates_for_FA[:,samples_to_consider]
smoothed_MN_firing_rates_for_FA = smoothed_MN_firing_rates_for_FA.T

# # Standardize the data
# smoothed_MN_firing_rates_for_FA = (smoothed_MN_firing_rates_for_FA - np.mean(smoothed_MN_firing_rates_for_FA, axis=0)) / np.std(smoothed_MN_firing_rates_for_FA, axis=0)

# fig, axs = plt.subplots(nb_pools, 1, figsize=(20, 3+(4*nb_pools)))
# if nb_pools == 1:
#     axs = [axs]
# mn_iter_i = -1
# mn_to_consider_idx = -1
# for pooli in range(nb_pools):
#     colormap_temp = cm.get_cmap(pool_cmap[pooli]) # Getting a smooth color blend from a given colormap
#     for mni in range(nb_motoneurons_per_pool):
#         mn_iter_i += 1
#         if mn_iter_i in idx_of_MUs_to_consider:
#             mn_to_consider_idx += 1
#             axs[nb_pools-pooli-1].plot(samples_to_consider, smoothed_MN_firing_rates_for_FA[:,mn_to_consider_idx], color=colormap_temp(mni / nb_motoneurons_per_pool), alpha=0.5)
#     axs[nb_pools-pooli-1].set_ylabel("Smoothed discharge rate (pps)")
#     axs[nb_pools-pooli-1].set_xlabel("Time (samples)")
#     axs[nb_pools-pooli-1].set_title(f"Pool #{pooli+1}")
# plt.suptitle(f"Standardized smoothed signals of only continuous MUs \n (dark = small MNs ; light = large MNs)")
# plt.tight_layout()
# new_filename = f'Standardized_smoothed_discharge_rates_only_continuous_MUs.png'
# save_file_path = os.path.join(new_directory, new_filename)
# plt.savefig(save_file_path)
# plt.show(fig)


In [ ]:
## DIMENSIONALITY REDUCTION

## FA ######
for factori in range(max_nb_of_factors_to_extract-1):
    fa = FactorAnalyzer(n_factors=factori+1, rotation='promax')
    fa.fit(smoothed_MN_firing_rates_for_FA)
    loadings.append(fa.loadings_)
    scores.append(fa.transform(smoothed_MN_firing_rates_for_FA))
    # get Rsquared value of FA's data reconstruction
    reconstructed_data_temp = np.dot(loadings[factori], scores[factori].T)
    Rsquared_per_MN_temp = []
    for mni in range(len(idx_of_MUs_to_consider)):
        Rsquared_per_MN_temp.append(
            (np.corrcoef(smoothed_MN_firing_rates_for_FA[:,mni],reconstructed_data_temp[mni])[0,1])
            **2) # get R² by calculating r^2
    reconstructed_Rsquared.append(np.mean(Rsquared_per_MN_temp))
reconstructed_Rsquared = np.insert(reconstructed_Rsquared, 0, 0.0)

In [ ]:
# Create surrogate random signal
# Only for the first pool

nb_shuffle_iter = 1
shuffled_smoothed_signal = list(range(nb_shuffle_iter))
shuffled_explained_variance = list(range(nb_shuffle_iter))
for shuffli in range(nb_shuffle_iter):
    print('Shuffling iteration ' + str(shuffli) + '...')
    shuffled_ISI = list(idx_of_MUs_to_consider)
    shuffled_binary_spike_trains = np.zeros(( len(idx_of_MUs_to_consider),
         int((duration_with_ignored_window/second)*fsamp))) # Initialize the binary spike train array
    # 'with ignored window' to allow for the shuffling to extend over the end of the signal (this will be ignored later) and to also avoid possible artifacts at the beginning
    shuffled_smoothed_signal[shuffli] = []
    for mni in range(len(idx_of_MUs_to_consider)):
        shuffled_ISI[mni] = np.random.permutation(diff(motoneurons_spike_trains[idx_of_MUs_to_consider[mni]]/second)) # list of random ISIs (one list per motoneuron)
        spikes_shuffled = cumsum(shuffled_ISI[mni])
        shuffled_binary_spike_trains[mni,
            np.round((spikes_shuffled-1+window_beginning_ignore)*fsamp).astype(int)] = 1
            #spikes_shuffled-1 because of offset due to 0-indexing
        shuffled_smoothed_signal[shuffli].append(filtfilt(HanningW, 1,
            shuffled_binary_spike_trains[mni,:] * fsamp))
        # ignoring the start and end windows
        shuffled_smoothed_signal[shuffli][-1] = shuffled_smoothed_signal[shuffli][-1][
            int(window_beginning_ignore * fsamp):-int(window_end_ignore * fsamp)]
    
    shuffled_smoothed_signal[shuffli] = np.array(shuffled_smoothed_signal[shuffli])
    plt.figure()
    plt.plot(shuffled_smoothed_signal[shuffli].T)
    plt.title("Sanity check of shuffled smoothed spike trains")

    # FA on shuffled signal #####
    loadings_shuffled = []
    scores_shuffled = []
    shuffled_explained_variance[shuffli] = list(range(max_nb_of_factors_to_extract-1))
    print('Calculating FA for shuffling iteration ' + str(shuffli) + '...')
    for factori in range(max_nb_of_factors_to_extract-1):
        fa_shuffled = FactorAnalyzer(n_factors=factori+1, rotation='promax')
        fa_shuffled.fit(shuffled_smoothed_signal[shuffli].T)
        loadings_shuffled.append(fa_shuffled.loadings_)
        scores_shuffled.append(fa_shuffled.transform(shuffled_smoothed_signal[shuffli].T))
        # get Rsquared value of FA's data reconstruction
        reconstructed_data_temp = np.dot(loadings_shuffled[factori], scores_shuffled[factori].T)
        Rsquared_per_MN_temp = []
        for mni in range(len(idx_of_MUs_to_consider)):
            Rsquared_per_MN_temp.append(
                (np.corrcoef(shuffled_smoothed_signal[shuffli][mni],reconstructed_data_temp[mni])[0,1])
                **2) # get R² by calculating r^2
        shuffled_explained_variance[shuffli][factori] = np.mean(Rsquared_per_MN_temp)
    shuffled_explained_variance[shuffli] = np.insert(shuffled_explained_variance[shuffli], 0, 0.0)

In [ ]:
plt.figure()
plt.plot(reconstructed_Rsquared, color=pool_colors_one[0], linewidth=2, label="Simulated data")
plt.plot(shuffled_explained_variance[0], color=pool_colors_one[1], linestyle='--', linewidth=2, label="Shuffled data")
plt.title(f"R² of reconstructed signals (pool #0)")
plt.xlabel('Nb of factors')
plt.ylabel('R²')
plt.legend()
plt.ylim([0,1])
new_filename = f'Rsquared_FA.png'
save_file_path = os.path.join(new_directory, new_filename)
# plt.savefig(save_file_path)
plt.show()

In [ ]:
# Calculate random slope at each point on the curve
# Get slope of variance explained for simulated VS random data
slope_VAF = []
slope_VAF_RMS = []
slope_VAF_random = []
slope_VAF_random_RMS = []
x_axis_components = np.arange(max_nb_of_factors_to_extract-2)+1
for componenti in range(max_nb_of_factors_to_extract-2):
    componenti = componenti+1
    # For simulated data
    #    # get slope
    x_temp = [componenti-1,componenti,componenti+1]
    y_temp = reconstructed_Rsquared[x_temp]
    slope_temp, intercept, r_value, p_value, std_err = linregress(x_temp, y_temp)
    slope_VAF.append(slope_temp)
    #    # get error of linear fit
    y_pred_temp = slope_temp * np.array(x_temp) + intercept
    residuals_temp = y_temp - y_pred_temp
    slope_VAF_RMS.append(np.sqrt(np.sum(residuals_temp**2)))
    # For shuffled data
    #    # get slope
    y_temp_random = shuffled_explained_variance[shuffli][x_temp]
    slope_temp, intercept, r_value, p_value, std_err = linregress(x_temp, y_temp_random)
    slope_VAF_random.append(slope_temp)
    #    # get error of linear fit
    y_pred_temp = slope_temp * np.array(x_temp) + intercept
    residuals_temp = y_temp_random - y_pred_temp
    slope_VAF_random_RMS.append(np.sqrt(np.sum(residuals_temp**2)))
plt.figure()
plt.plot(x_axis_components,slope_VAF)
plt.plot(x_axis_components,slope_VAF_random)
plt.ylabel("Slope")
plt.xlabel("Nb of components")
plt.title(f"Slope values (pool 0)")

nb_of_factors_above_which_Rsquared_is_below_Rsquared_of_shuffled_data = []
for componenti in range(len(slope_VAF)):
    if slope_VAF[componenti] <= slope_VAF_random[componenti]: #np.mean(slope_VAF_random[componenti])
        nb_of_factors_above_which_Rsquared_is_below_Rsquared_of_shuffled_data = componenti
        break
plt.axvline(x = nb_of_factors_above_which_Rsquared_is_below_Rsquared_of_shuffled_data, color = 'crimson', linewidth=2)
legend(["Simulated data slope of R² curve",
        "Shuffled data slope of R² curve",
        "Nb of components AFTER which slope of R² <= slope of surrogate (random) R²"])
new_filename = 'Additional_Rsquared_per_new_factor.png'
save_file_path = os.path.join(new_directory, new_filename)
plt.savefig(save_file_path)
plt.show(fig)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(reconstructed_Rsquared)
ax.plot(shuffled_explained_variance[0], ls='--') # plot line for randomized signal
ax.axvline(nb_excitatory_inputs_per_pool, ls='-', c='C2', lw=10, alpha = 0.5) # nb of excitatory inputs
ax.axvline(nb_inhibitory_inputs_per_pool, ls='-', c='C4', lw=10, alpha = 0.5) # nb of inhibiitory inputs
ax.axvline(nb_of_factors_above_which_Rsquared_is_below_Rsquared_of_shuffled_data, ls='--', c='crimson', lw=2) # Slope of true variance explained < slope of variance explained for noise/random data
# ax.axvline(nb_of_factors_above_which_curvature_is_below_curvature_of_shuffled_data, ls='--', c='magenta', lw=3) # Slope of true variance explained < slope of variance explained for noise/random data
ax.set_xlabel('Nb of components')
ax.set_ylabel('Variance explained / R²')
# ax.set_xlim(0, min([nb_motoneurons,10]))
ax.set_ylim(0, 1)
plt.legend(["Simulation","Randomized (noise)","Nb of excit. components","Nb of inhib. components",
        "Nb of components AFTER which slope of R² <= slope of surrogate (random) R²",
        "Nb of components AFTER which curvature of R² curve <= curvature of surrogate (random) R² curve"])
plt.title(f"R² with factor analysis (pool 0)")
new_filename = 'Rsquared_curve.png'
save_file_path = os.path.join(new_directory, new_filename)
plt.savefig(save_file_path)
plt.show(fig)

In [194]:
# Save the R2 values of the reconstructed data from FA as a csv file
vectors_results_to_save = [nb_of_factors_above_which_Rsquared_is_below_Rsquared_of_shuffled_data,
                           len(np.where(valid_MU_idx)[0]),
                            np.mean(mean_DR_per_MN[valid_MU_idx]),
                            np.std(mean_DR_per_MN[valid_MU_idx]),
                            np.mean(std_DR_per_MN[valid_MU_idx]),
                            np.std(mean_DR_per_MN[valid_MU_idx]),
                            mean_force_during_contraction,
                            np.std(normalized_force)
                            ]
column_names_of_results_to_save = ['Nb_factors',
                                   'Nb_valid_MUs',
                                   'Mean_of_mean_DRs_of_valid_MUs',
                                   'STD_of_mean_DRs_of_valid_MUs',
                                   'mean_of_STD_of_DRs_of_valid_MUs',
                                   'STD_of_STD_of_DRs_of_valid_MUs',
                                   'Force_mean',
                                   'Force_normalized_STD'] # How much the force fluctuates relative to the mean force
# Save the R2 values of the reconstructed data from FA as a CSV file
df = pd.DataFrame([vectors_results_to_save], columns=column_names_of_results_to_save)
# Save as a CSV file
new_filename = '_Results_FA_nb_of_factors_and_MUs_params.csv'
df.to_csv( os.path.join(new_directory, new_filename), index=False, header=True)

# Save the R2 values of the reconstructed data from FA as a CSV file
df = pd.DataFrame(reconstructed_Rsquared, columns=["R2 Value"])
df.index.name = "Nb_of_factors"  # Index column name
# Save as a CSV file
new_filename = 'FA_reconstructed_R2.csv'
df.to_csv( os.path.join(new_directory, new_filename), index=True, header=True)

if save_binary_dsicharge_mat_as_csv:
    # Save binary spike train as csv file
    df = pd.DataFrame(motoneurons_binary_spike_trains)
    df.index.name = "MN_index"  # Index column name
    # Save as a CSV file
    new_filename = 'motoneurons_binary_spike_trains.csv'
    df.to_csv( os.path.join(new_directory, new_filename), index=True, header=True)

# Save the valid MU indices (continuously firing MUs) as a CSV file
df = pd.DataFrame(np.where(valid_MU_idx)[0], columns=["Valid MUs indices"])
# Save as a CSV file
new_filename = 'idx_of_continuously_firing_MUs.csv'
df.to_csv( os.path.join(new_directory, new_filename), index=False, header=True)
